# train_sanity.ipynb — Slakh Sanity Training Run

**Prerequisites:** CT1-CT5 all PASSED in `smoke_test.ipynb`.

| Cell | Action |
|------|--------|
| 1    | Drive mount + git clone + pip install + constants |
| 1.5  | B1: Download Slakh2100 redux + filter band-style tracks → SLAKH_RAW |
| 2    | Run `slakh_adapter.py` on downloaded Slakh tracks → manifest |
| 3    | `split_dataset.py` → train/val/test CSVs |
| 4    | **Training** (30k steps, overnight) — supports `--resume_from auto` |
| 5    | **Loss curves** (matplotlib) — reads TensorBoard event files, plots train/val loss + LR |
| 6    | Gate check: val_loss in [0.15, 0.30] + listen-check generated samples |
| 7    | **Held-out val-set F1 eval** — vocode 1 val segment + Basic-Pitch + compute F1 (gate ≥ 0.30) |

**NOTE — validation during training vs. post-training evaluation:**
- `val/loss` is computed every 500 steps throughout training (`compute_val_loss()` in train.py) — no separate run needed.
- Cell 7 runs *inference* on one val segment (different from loss: it generates audio and scores it with F1).
- The **test set** is never used until final evaluation for the paper/presentation.

**Slakh gate (REMAINING_WORKPLAN.md STEP 3) — ALL must pass before proceeding to Israeli training:**
- Val loss (best) ∈ [0.15, 0.30]  — verified by Cell 5 / Cell 6
- Vocoded sample is recognizably musical (subjective listen-check in Cell 7)
- Held-out F1 ≥ 0.30  — computed in Cell 7


---

### §37 cell-header convention

Every code cell in this notebook is preceded by a markdown header that
follows the project standard (ENGINEERING_DECISIONS §37, see
[colab/README.md](README.md)):

> **What this does.** plain-English summary.  
> **Inputs.** files / variables / env read.  
> **Outputs.** files / variables / state written.  
> **Action required.** anything the user must edit, click, approve, or run
> elsewhere. Prefix the heading with **⚠** when the action requires switching
> machines (e.g. F1 backfill needs `basic_pitch_env` locally).  
> **Runtime.** order-of-magnitude (seconds / minutes / hours).

Stub headers below carry `TODO` placeholders — fill them in when you next
edit the cell. `colab/postprocessing.ipynb` is the reference implementation.


<!-- §37-stub -->
## Cell 1 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 1 — Environment setup
# (Same as CT1; safe to re-run if already done in smoke_test)
# ============================================================

from google.colab import drive, userdata
drive.mount('/content/drive')

# ── GitHub auth via Colab secret 'GITHUB_TOKEN' ──────────────────────────
# In Colab: Secrets (key icon) → find 'GITHUB_TOKEN' → toggle Notebook access ON
# Token needs 'repo' scope (classic PAT)
import os, subprocess

_token = userdata.get('GITHUB_TOKEN')   # raises SecretNotFoundError if missing/access off
assert _token, "GITHUB_TOKEN secret is empty — check Colab Secrets panel"
REPO_URL = f'https://{_token}@github.com/galgeva1/MusicProject.git'
print(f'Token loaded: {_token[:4]}...{_token[-4:]}  (len={len(_token)})')

REPO_DIR = '/content/MusicProject'
if not os.path.exists(REPO_DIR):
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError('git clone failed — check token scope and repo name')
else:
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'],
                            capture_output=True, text=True)
    print(result.stdout)

%cd {REPO_DIR}

# Use requirements_colab.txt — skips torch (pre-installed with CUDA) and
# uses loose version pins compatible with Colab's Python 3.12
!pip install -q -r requirements_colab.txt
!pip install -q basic-pitch

# Path constants
DRIVE_ROOT       = '/content/drive/MyDrive/MusicProject'
DATA_ROOT        = f'{DRIVE_ROOT}/MusicProjectData'
SLAKH_RAW        = f'{DRIVE_ROOT}/slakh_raw'
SLAKH_PROCESSED  = f'{DRIVE_ROOT}/slakh_processed'
MANIFEST_DIR     = f'{DRIVE_ROOT}/data'
RUN_NAME         = 'slakh_v2'                # ← change per experiment (e.g. 'israeli_v1')
MAX_STEPS        = 100_000   # ← set this before each run; controls training length AND
                             #   LR schedule shape (flat until decay_start_frac=0.9
                             #   i.e. step 90k, then linear decay to lr_min)
CKPT_DIR         = f'{DRIVE_ROOT}/checkpoints/{RUN_NAME}'
LOG_DIR          = f'{DRIVE_ROOT}/logs/{RUN_NAME}'
SLAKH_MANIFEST   = f'{MANIFEST_DIR}/slakh_manifest.csv'
SPLITS_DIR       = f'{MANIFEST_DIR}/slakh_splits'
TRAIN_CSV        = f'{SPLITS_DIR}/train.csv'
VAL_CSV          = f'{SPLITS_DIR}/val.csv'
TEST_CSV         = f'{SPLITS_DIR}/test.csv'

import pathlib
for d in [DATA_ROOT, SLAKH_RAW, SLAKH_PROCESSED, MANIFEST_DIR, CKPT_DIR, LOG_DIR]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# Verify imports
import sys
sys.path.insert(0, REPO_DIR)
from preprocessing.slakh_adapter import adapt_slakh_split
from preprocessing.split_dataset import split_dataset
print('Imports OK')
print(f'Run name:      {RUN_NAME}')
print(f'Slakh raw dir: {SLAKH_RAW}')

print(f'Checkpoints:   {CKPT_DIR}')
print(f'Val CSV:       {VAL_CSV}')

<!-- §37-stub -->
## Cell 2 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 1.5 — B1: Download Slakh2100 redux + extract band-style subset
# ============================================================
#
# NOTE: Slakh metadata.yaml has NO genre field.
#       We proxy rock/pop by requiring each track to have ALL of:
#         Piano, Guitar, Bass, Drums  (mirrors band_defs/rock_band.json)
#
# What this cell does (two-pass streaming over the 104 GB tar):
#   Pass 1: reads every train-split metadata.yaml (tiny, no disk write)
#            → identifies 'band-style' tracks (~1289 expected)
#   Pass 2: extracts mix.flac + MIDI/ + metadata.yaml for ALL qualifying
#            tracks directly into SLAKH_RAW on Drive
#
# Disk usage : ~104 GB in /tmp (deleted at end) + ~107 GB in SLAKH_RAW
# Expected runtime : ~4-6 h on Colab Pro (mostly download + 2 tar scans)
# Idempotent : skipped if SLAKH_RAW already has >= MIN_EXISTING track dirs.
# ============================================================

import pathlib, yaml, tarfile, random

TAR_URL    = 'https://zenodo.org/records/4599666/files/slakh2100_flac_redux.tar.gz?download=1'
TAR_PATH   = pathlib.Path('/tmp/slakh2100_flac_redux.tar.gz')
MIN_EXISTING = 1000   # skip download if Drive already has this many track dirs

# Instrument-class filter: Slakh rock_band definition
# mirrors band_defs/rock_band.json (key='class', all four required)
ROCK_BAND_CLASSES = {'Piano', 'Guitar', 'Bass', 'Drums'}

slakh_raw_path = pathlib.Path(SLAKH_RAW)
existing = [d for d in slakh_raw_path.iterdir() if d.is_dir()] if slakh_raw_path.exists() else []

if len(existing) >= MIN_EXISTING:
    print(f'B1 SKIP — SLAKH_RAW already has {len(existing)} track dirs (>= {MIN_EXISTING}).')
else:
    # ── 1. Download ──────────────────────────────────────────────────
    if not TAR_PATH.exists() or TAR_PATH.stat().st_size < 1_000_000:
        print('Downloading Slakh2100 redux (~104 GB) — ~4-6 h ...')
        !wget -q --show-progress --continue '{TAR_URL}' -O '{TAR_PATH}'
    else:
        print(f'Archive present: {TAR_PATH} ({TAR_PATH.stat().st_size/1e9:.1f} GB)')

    # ── 2. Detect archive prefix (e.g. 'slakh2100_flac_redux/train/') ───
    print('\nDetecting archive structure ...')
    train_prefix = ''
    with tarfile.open(str(TAR_PATH), 'r|gz') as tar:
        for i, member in enumerate(tar):
            if i > 500:
                break
            parts = pathlib.PurePosixPath(member.name).parts
            if len(parts) >= 4 and parts[-2] == 'MIDI' and parts[-1].endswith('.mid'):
                train_prefix = '/'.join(parts[:-3]) + '/'
                print(f'  train prefix: "{train_prefix}"')
                break
    assert train_prefix, 'Could not detect archive structure — check tar layout'

    # ── 3. Pass 1 — scan metadata.yaml, find ALL band-style tracks ──────────
    print('\nPass 1 — scanning metadata.yaml (streaming, ~20 min) ...')
    band_tracks = []
    with tarfile.open(str(TAR_PATH), 'r|gz') as tar:
        for member in tar:
            if not member.isfile():
                continue
            if not member.name.startswith(train_prefix):
                continue
            parts = pathlib.PurePosixPath(member.name).parts
            if parts[-1] != 'metadata.yaml':
                continue
            track_id = parts[-2]
            f = tar.extractfile(member)
            if not f:
                continue
            try:
                meta = yaml.safe_load(f.read().decode('utf-8', errors='replace'))
                stems = meta.get('stems', {}) if isinstance(meta, dict) else {}
                classes = {v.get('inst_class', '') for v in stems.values()
                           if isinstance(v, dict)}
                if ROCK_BAND_CLASSES.issubset(classes):
                    band_tracks.append(track_id)
            except Exception:
                pass
    print(f'  Band-style tracks found: {len(band_tracks)}')
    assert band_tracks, 'No band-style tracks found — check archive structure'

    # ── 4. Select ALL qualifying tracks ─────────────────────────────────────
    selected = set(band_tracks)
    print(f'  Extracting ALL {len(selected)} qualifying tracks')
    print(f'  Sample IDs: {sorted(selected)[:5]} ...')

    # ── 5. Pass 2 — extract selected tracks into SLAKH_RAW ──────────────────
    print(f'\nPass 2 — extracting {len(selected)} tracks (~20 min) ...')
    slakh_raw_path.mkdir(parents=True, exist_ok=True)
    n_extracted = 0
    with tarfile.open(str(TAR_PATH), 'r|gz') as tar:
        for member in tar:
            if not member.isfile():
                continue
            if not member.name.startswith(train_prefix):
                continue
            rel       = member.name[len(train_prefix):]  # e.g. 'Track00001/mix.flac'
            rel_parts = pathlib.PurePosixPath(rel).parts
            if not rel_parts:
                continue
            track_id = rel_parts[0]
            if track_id not in selected:
                continue
            rest = rel_parts[1:]
            if not rest:
                continue
            # Keep only: mix.flac, metadata.yaml, MIDI/*.mid
            if rest[-1] in ('mix.flac', 'metadata.yaml'):
                pass
            elif len(rest) == 2 and rest[0] == 'MIDI' and rest[1].endswith('.mid'):
                pass
            else:
                continue  # skip stems and other files
            dest = slakh_raw_path / pathlib.Path(*rel_parts)
            dest.parent.mkdir(parents=True, exist_ok=True)
            f = tar.extractfile(member)
            if f:
                dest.write_bytes(f.read())
                n_extracted += 1
    print(f'  Extracted {n_extracted} files.')

    # ── 6. Cleanup ──────────────────────────────────────────────────────────
    print(f'\nRemoving {TAR_PATH} to free /tmp space ...')
    TAR_PATH.unlink(missing_ok=True)

    # ── 7. Verify ───────────────────────────────────────────────────────────
    track_dirs = sorted(d for d in slakh_raw_path.iterdir() if d.is_dir())
    mix_count  = sum(1 for d in track_dirs if (d / 'mix.flac').exists())
    print(f'\nSLAKH_RAW: {len(track_dirs)} track dirs, {mix_count} with mix.flac')
    assert len(track_dirs) >= MIN_EXISTING, f'B1 FAILED: got {len(track_dirs)} tracks, need >= {MIN_EXISTING}'
    print('B1 PASSED')


<!-- §37-stub -->
## Cell 3 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 1.75 — B2: Quality Filter
# ============================================================
# Automated per-track checks on every track in SLAKH_RAW:
#   1. mix.flac exists and loads without error
#   2. Duration >= 180s (3 min, enough for 36 × 5s segments)
#   3. RMS > 0.005 on first 30s (not silent/corrupted)
#   4. At least one MIDI file with total note count > 50
#   5. MIDI end time within ±20% of audio duration (alignment)
#
# Passing tracks → DRIVE_ROOT/slakh_quality_passed.txt
# Failing tracks → moved to slakh_rejected/ (sibling of slakh_raw/)
# Idempotent: skips if quality_passed.txt already exists.
# ============================================================

import pathlib, soundfile as sf, numpy as np, pretty_midi, shutil

SLAKH_REJECTED      = str(pathlib.Path(SLAKH_RAW).parent / 'slakh_rejected')
QUALITY_PASSED_FILE = str(pathlib.Path(DRIVE_ROOT) / 'slakh_quality_passed.txt')

MIN_DURATION_S  = 180.0   # 3 minutes
MIN_RMS         = 0.005   # noise floor
MIN_MIDI_NOTES  = 50      # total notes across all MIDI files
MIDI_ALIGN_TOL  = 0.20    # MIDI end time within ±20% of audio duration

if pathlib.Path(QUALITY_PASSED_FILE).exists():
    passed = [l.strip() for l in open(QUALITY_PASSED_FILE) if l.strip()]
    print(f'B2 SKIP — quality_passed.txt already exists ({len(passed)} tracks).')
else:
    slakh_raw_path = pathlib.Path(SLAKH_RAW)
    rejected_path  = pathlib.Path(SLAKH_REJECTED)
    rejected_path.mkdir(parents=True, exist_ok=True)

    track_dirs = sorted(d for d in slakh_raw_path.iterdir() if d.is_dir())
    print(f'Checking {len(track_dirs)} tracks ...\n')

    passed, failed = [], []
    rows = []

    for track_dir in track_dirs:
        tid     = track_dir.name
        reasons = []

        # 1. Audio file
        mix_path = track_dir / 'mix.flac'
        if not mix_path.exists():
            reasons.append('no mix.flac')
            failed.append(tid); rows.append((tid, 'FAIL', ', '.join(reasons))); continue

        # 2. Duration
        try:
            info     = sf.info(str(mix_path))
            duration = info.frames / info.samplerate
            if duration < MIN_DURATION_S:
                reasons.append(f'short {duration:.0f}s')
        except Exception as e:
            reasons.append(f'audio error: {e}')
            failed.append(tid); rows.append((tid, 'FAIL', ', '.join(reasons))); continue

        # 3. RMS (first 30s only for speed)
        try:
            data, _ = sf.read(str(mix_path), frames=int(info.samplerate * 30),
                              dtype='float32', always_2d=False)
            if data.ndim > 1:
                data = data.mean(axis=1)
            rms = float(np.sqrt(np.mean(data ** 2)))
            if rms < MIN_RMS:
                reasons.append(f'silent rms={rms:.4f}')
        except Exception as e:
            reasons.append(f'rms error: {e}')

        # 4+5. MIDI
        midi_dir   = track_dir / 'MIDI'
        midi_files = list(midi_dir.glob('*.mid')) if midi_dir.exists() else []
        if not midi_files:
            reasons.append('no MIDI files')
        else:
            total_notes, midi_end = 0, 0.0
            for mf in midi_files:
                try:
                    pm = pretty_midi.PrettyMIDI(str(mf))
                    for inst in pm.instruments:
                        total_notes += len(inst.notes)
                    midi_end = max(midi_end, pm.get_end_time())
                except Exception:
                    pass
            if total_notes < MIN_MIDI_NOTES:
                reasons.append(f'few notes ({total_notes})')
            if midi_end > 0 and duration > 0:
                ratio = abs(midi_end - duration) / duration
                if ratio > MIDI_ALIGN_TOL:
                    reasons.append(f'misaligned midi={midi_end:.0f}s audio={duration:.0f}s')

        if reasons:
            failed.append(tid)
            rows.append((tid, 'FAIL', ', '.join(reasons)))
            shutil.move(str(track_dir), str(rejected_path / tid))
        else:
            passed.append(tid)
            rows.append((tid, 'PASS', f'{duration:.0f}s rms={rms:.3f}'))

    with open(QUALITY_PASSED_FILE, 'w') as f:
        f.write('\n'.join(passed))

    print(f'{"Track":<14} {"Result":<6} Details')
    print('-' * 65)
    for tid, result, detail in rows:
        print(f'{tid:<14} {result:<6} {detail}')
    print(f'\nPASSED: {len(passed)} / {len(track_dirs)}  |  REJECTED: {len(failed)}')
    assert len(passed) >= 40, f'B2 FAILED: only {len(passed)} tracks passed quality check'
    print('B2 PASSED')


<!-- §37-stub -->
## Cell 4 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 1.85 — B3: Style / Tempo Filter
# ============================================================
# Reads quality_passed.txt, estimates BPM for each track using
# librosa.beat.beat_track on the first 60s of audio.
# Keeps tracks with estimated BPM in [BPM_MIN, BPM_MAX] to
# exclude extreme metal (>170) and ambient/slow (< 70).
#
# Output : DRIVE_ROOT/slakh_style_candidates.txt
# Idempotent: skips if file already exists.
# ============================================================

import pathlib, soundfile as sf, numpy as np
import librosa

BPM_MIN = 70
BPM_MAX = 170

QUALITY_PASSED_FILE   = str(pathlib.Path(DRIVE_ROOT) / 'slakh_quality_passed.txt')
STYLE_CANDIDATES_FILE = str(pathlib.Path(DRIVE_ROOT) / 'slakh_style_candidates.txt')

if pathlib.Path(STYLE_CANDIDATES_FILE).exists():
    candidates = [l.strip() for l in open(STYLE_CANDIDATES_FILE) if l.strip()]
    print(f'B3 SKIP — style_candidates.txt already exists ({len(candidates)} tracks).')
else:
    passed = [l.strip() for l in open(QUALITY_PASSED_FILE) if l.strip()]
    print(f'Estimating BPM for {len(passed)} tracks ...\n')

    candidates, rejected = [], []
    for tid in passed:
        mix_path = pathlib.Path(SLAKH_RAW) / tid / 'mix.flac'
        try:
            info = sf.info(str(mix_path))
            frames_60s = int(info.samplerate * 60)
            data, sr = sf.read(str(mix_path), frames=frames_60s, dtype='float32', always_2d=False)
            if data.ndim > 1:
                data = data.mean(axis=1)
            if sr != 22050:
                data = librosa.resample(data, orig_sr=sr, target_sr=22050)
                sr = 22050
            tempo, _ = librosa.beat.beat_track(y=data, sr=sr)
            bpm = float(tempo)
            if BPM_MIN <= bpm <= BPM_MAX:
                candidates.append(tid)
                tag = 'KEEP'
            else:
                rejected.append(tid)
                tag = f'SKIP (BPM={bpm:.0f})'
            print(f'  {tid}: {bpm:.1f} BPM → {tag}')
        except Exception as e:
            print(f'  {tid}: ERROR {e} → SKIP')
            rejected.append(tid)

    with open(STYLE_CANDIDATES_FILE, 'w') as f:
        f.write('\n'.join(candidates))

    print(f'\nKEPT: {len(candidates)} / {len(passed)}  |  BPM-filtered out: {len(rejected)}')
    assert len(candidates) >= 30, f'B3 FAILED: only {len(candidates)} tracks in BPM range [{BPM_MIN}, {BPM_MAX}]'
    print('B3 PASSED')


<!-- §37-stub -->
## Cell 5 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:

# ============================================================
# Cell 1.9 — B4: Hearing Test (interactive, run in Colab)
# ============================================================
# Plays 30s clips inline. For each track: Keep / Reject / Unsure.
# Tracks are shown in batches of 10 with Prev/Next navigation.
#
# RESUME SUPPORT:
#   - Every "Save" click writes slakh_decisions_checkpoint.json to Drive
#     (keeps all three states: keep / reject / unsure).
#   - On re-run, the checkpoint is loaded automatically and the widget
#     starts at the first undecided batch (all-unsure tracks).
#   - "Finalize & Write slakh_curated.txt" is only shown once all
#     tracks have a keep/reject decision (no remaining unsure).
#   - slakh_curated.txt is written only when you click Finalize.
# ============================================================

import pathlib, json, soundfile as sf, numpy as np
import ipywidgets as widgets
from IPython.display import display, Audio, HTML, clear_output

STYLE_CANDIDATES_FILE = str(pathlib.Path(DRIVE_ROOT) / 'slakh_style_candidates.txt')
CHECKPOINT_FILE       = str(pathlib.Path(DRIVE_ROOT) / 'slakh_decisions_checkpoint.json')
CURATED_FILE          = str(pathlib.Path(DRIVE_ROOT) / 'slakh_curated.txt')
CLIP_START_S          = 30   # skip intro, start at 30s
CLIP_DURATION_S       = 30
BATCH_SIZE            = 10

# ── Load candidate list ──────────────────────────────────────
candidates = [l.strip() for l in open(STYLE_CANDIDATES_FILE) if l.strip()]
print(f'Loaded {len(candidates)} style candidates.')

# ── Load or initialise decisions ─────────────────────────────
if pathlib.Path(CHECKPOINT_FILE).exists():
    decisions = json.loads(pathlib.Path(CHECKPOINT_FILE).read_text())
    # Ensure all candidates are present (handles new candidates added after a partial run)
    for tid in candidates:
        decisions.setdefault(tid, 'unsure')
    n_decided = sum(1 for d in decisions.values() if d != 'unsure')
    n_keep    = sum(1 for d in decisions.values() if d == 'keep')
    print(f'Resuming from checkpoint: {n_decided}/{len(candidates)} decided '
          f'({n_keep} keep, {n_decided - n_keep} reject, {len(candidates) - n_decided} unsure remaining).')
else:
    decisions = {tid: 'unsure' for tid in candidates}
    print('No checkpoint found — starting fresh.')

# ── Find first undecided batch ───────────────────────────────
undecided_indices = [i for i, tid in enumerate(candidates) if decisions[tid] == 'unsure']
first_undecided_batch = (undecided_indices[0] // BATCH_SIZE) if undecided_indices else 0

n_batches = (len(candidates) + BATCH_SIZE - 1) // BATCH_SIZE
batch_idx = [first_undecided_batch]

# ── Helpers ──────────────────────────────────────────────────
def load_clip(track_id):
    mix_path = pathlib.Path(SLAKH_RAW) / track_id / 'mix.flac'
    info     = sf.info(str(mix_path))
    start    = int(info.samplerate * CLIP_START_S)
    frames   = int(info.samplerate * CLIP_DURATION_S)
    data, sr = sf.read(str(mix_path), start=start, frames=frames, dtype='float32', always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    return data, sr

def save_checkpoint():
    pathlib.Path(CHECKPOINT_FILE).write_text(json.dumps(decisions, indent=2))

def progress_html():
    n_keep   = sum(1 for d in decisions.values() if d == 'keep')
    n_rej    = sum(1 for d in decisions.values() if d == 'reject')
    n_unsure = sum(1 for d in decisions.values() if d == 'unsure')
    pct      = int(100 * (n_keep + n_rej) / len(decisions))
    return (f'<span style="font-size:13px">✅ keep: <b>{n_keep}</b> &nbsp;'
            f'❌ reject: <b>{n_rej}</b> &nbsp;'
            f'❓ unsure: <b>{n_unsure}</b> &nbsp;|&nbsp; '
            f'<b>{pct}%</b> decided</span>')

# ── Widget builder ────────────────────────────────────────────
def make_batch_widget(batch):
    items = []
    for tid in batch:
        # Wrap Audio in Output widget (Audio is not an ipywidgets Widget)
        audio_out = widgets.Output()
        try:
            data, sr = load_clip(tid)
            with audio_out:
                display(Audio(data, rate=sr, autoplay=False))
        except Exception as e:
            with audio_out:
                print(f'Error loading {tid}: {e}')

        cur     = decisions[tid]
        lbl     = widgets.HTML(f'<b style="font-size:14px">{tid}</b>')
        status  = widgets.Label(f'[{cur}]')
        b_keep  = widgets.Button(description='✓ Keep',   button_style='success', layout=widgets.Layout(width='110px'))
        b_rej   = widgets.Button(description='✗ Reject', button_style='danger',  layout=widgets.Layout(width='110px'))
        b_uns   = widgets.Button(description='? Unsure', button_style='warning', layout=widgets.Layout(width='110px'))

        # Highlight current decision
        if cur == 'keep':   b_keep.icon = 'check'
        elif cur == 'reject': b_rej.icon = 'times'

        def _cb(t, s, st):
            def cb(b):
                decisions[t] = s
                st.value = f'[{s}]'
                prog_label.value = progress_html()
            return cb
        b_keep.on_click(_cb(tid, 'keep',   status))
        b_rej.on_click( _cb(tid, 'reject', status))
        b_uns.on_click( _cb(tid, 'unsure', status))
        items.append(widgets.VBox(
            [lbl, audio_out, widgets.HBox([b_keep, b_rej, b_uns, status])],
            layout=widgets.Layout(border='1px solid #ccc', padding='8px', margin='4px')
        ))
    return widgets.VBox(items)

# ── Layout ────────────────────────────────────────────────────
main_out   = widgets.Output()
nav_out    = widgets.Output()
prog_label = widgets.HTML(value=progress_html())

def show_batch(idx):
    batch = candidates[idx * BATCH_SIZE : (idx + 1) * BATCH_SIZE]
    undecided_in_batch = sum(1 for tid in batch if decisions[tid] == 'unsure')
    with main_out:
        clear_output(wait=True)
        display(HTML(
            f'<h3>Batch {idx+1}/{n_batches} &nbsp;'
            f'(tracks {idx*BATCH_SIZE+1}–{min((idx+1)*BATCH_SIZE, len(candidates))} of {len(candidates)}) &nbsp;'
            f'— {undecided_in_batch} undecided in this batch</h3>'
        ))
        display(make_batch_widget(batch))

def on_prev(b):
    if batch_idx[0] > 0:
        batch_idx[0] -= 1
        show_batch(batch_idx[0])

def on_next(b):
    if batch_idx[0] < n_batches - 1:
        batch_idx[0] += 1
        show_batch(batch_idx[0])

def on_save(b):
    save_checkpoint()
    n_keep    = sum(1 for d in decisions.values() if d == 'keep')
    n_decided = sum(1 for d in decisions.values() if d != 'unsure')
    with nav_out:
        clear_output(wait=True)
        print(f'✔ Checkpoint saved → {CHECKPOINT_FILE}')
        print(f'  {n_decided}/{len(decisions)} decided ({n_keep} keep). '
              f'{len(decisions) - n_decided} still unsure.')
        if len(decisions) - n_decided == 0:
            print('  All tracks decided! Click "Finalize" to write slakh_curated.txt.')

def on_finalize(b):
    n_unsure = sum(1 for d in decisions.values() if d == 'unsure')
    if n_unsure > 0:
        with nav_out:
            clear_output(wait=True)
            print(f'⚠ {n_unsure} tracks still unsure. Save checkpoint first and finish tagging, '
                  f'or they will be excluded from slakh_curated.txt.')
    kept = [t for t in candidates if decisions[t] == 'keep']
    pathlib.Path(CURATED_FILE).write_text('\n'.join(kept))
    save_checkpoint()
    with nav_out:
        clear_output(wait=True)
        print(f'✔ slakh_curated.txt written → {len(kept)} tracks → {CURATED_FILE}')
        print('  Run Cell 1.95 to load the curated list.')

b_prev     = widgets.Button(description='◀ Prev',      layout=widgets.Layout(width='100px'))
b_next     = widgets.Button(description='Next ▶',      layout=widgets.Layout(width='100px'))
b_save     = widgets.Button(description='💾 Save checkpoint', button_style='primary',  layout=widgets.Layout(width='200px'))
b_finalize = widgets.Button(description='✅ Finalize → curated.txt', button_style='success', layout=widgets.Layout(width='230px'))

b_prev.on_click(on_prev)
b_next.on_click(on_next)
b_save.on_click(on_save)
b_finalize.on_click(on_finalize)

show_batch(batch_idx[0])
display(prog_label)
display(main_out)
display(widgets.HBox([b_prev, b_next, b_save, b_finalize]))
display(nav_out)


<!-- §37-stub -->
## Cell 6 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 1.92 — Recover: write slakh_curated.txt from checkpoint
# ============================================================
# Run this cell if slakh_curated.txt is missing but
# slakh_decisions_checkpoint.json already exists on Drive
# (i.e. your partner saved the checkpoint but didn't click Finalize).
#
# It reads all 'keep' decisions from the checkpoint and writes
# slakh_curated.txt exactly as the Finalize button would have.
# ============================================================

import pathlib, json

CHECKPOINT_FILE = str(pathlib.Path(DRIVE_ROOT) / 'slakh_decisions_checkpoint.json')
CURATED_FILE    = str(pathlib.Path(DRIVE_ROOT) / 'slakh_curated.txt')

assert pathlib.Path(CHECKPOINT_FILE).exists(), (
    f'Checkpoint not found at {CHECKPOINT_FILE}.\n'
    'Neither the checkpoint nor curated.txt exist — the hearing test was not saved at all.\n'
    'Run Cell 1.9 to redo the hearing test.'
)

decisions = json.loads(pathlib.Path(CHECKPOINT_FILE).read_text())

n_keep   = sum(1 for d in decisions.values() if d == 'keep')
n_reject = sum(1 for d in decisions.values() if d == 'reject')
n_unsure = sum(1 for d in decisions.values() if d == 'unsure')
print(f'Checkpoint summary:')
print(f'  keep    : {n_keep}')
print(f'  reject  : {n_reject}')
print(f'  unsure  : {n_unsure}  ← these are excluded from curated list')
print(f'  total   : {len(decisions)}')

kept = [tid for tid, d in sorted(decisions.items()) if d == 'keep']

if pathlib.Path(CURATED_FILE).exists():
    print(f'\nslakh_curated.txt already exists — overwriting with checkpoint keep decisions.')

pathlib.Path(CURATED_FILE).write_text('\n'.join(kept))
print(f'\n✓ Wrote {len(kept)} track IDs → {CURATED_FILE}')
if n_unsure > 0:
    print(f'  NOTE: {n_unsure} unsure tracks were excluded.')
    print(f'  If you want to review them, run Cell 1.9 (it will resume from checkpoint).')
print('  → Run Cell 1.95 next.')


<!-- §37-stub -->
## Cell 7 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 1.95 — B5: Load curated track list
# ============================================================
# Reads slakh_curated.txt written by the hearing test (Cell 1.9).
# Sets CURATED_TRACKS for use by Cell 2 (slakh_adapter).
# Run this cell after completing the hearing test and saving.
# ============================================================

import pathlib

CURATED_FILE = str(pathlib.Path(DRIVE_ROOT) / 'slakh_curated.txt')

assert pathlib.Path(CURATED_FILE).exists(), (
    f'slakh_curated.txt not found at {CURATED_FILE}.\n'
    'Complete the hearing test in Cell 1.9 and click "Save Results".'
)

CURATED_TRACKS = [l.strip() for l in open(CURATED_FILE) if l.strip()]
assert len(CURATED_TRACKS) >= 50, (
    f'Only {len(CURATED_TRACKS)} curated tracks — need >= 50 for training.\n'
    'Go back to Cell 1.9 and keep more tracks, or relax BPM filter in Cell 1.85.'
)

print(f'B5 PASSED — {len(CURATED_TRACKS)} curated tracks loaded.')
total_min = len(CURATED_TRACKS) * 3  # rough estimate at 3 min/track
print(f'Estimated audio: ~{total_min // 60}h {total_min % 60}min ({len(CURATED_TRACKS)} tracks × 3 min avg)')
print(f'Estimated segments: ~{len(CURATED_TRACKS) * 36} (at 5s each, no overlap)')


<!-- §37-stub -->
## Cell 8 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 1.97 — B5 exact duration gate
# ============================================================
# Reads actual audio duration from mix.flac for every curated track.
# Target: >= 3 h total (mirrors the Israeli dataset target).
# Runs in ~1 min (soundfile reads headers only, no decoding).
# ============================================================

import soundfile as sf
import pathlib

MIN_HOURS = 3.0   # change to 6.0 if you want the upper gate too

total_s = 0.0
missing = []
n_ok    = 0

for tid in CURATED_TRACKS:
    mix = pathlib.Path(SLAKH_RAW) / tid / 'mix.flac'
    if not mix.exists():
        missing.append(tid)
        continue
    try:
        info     = sf.info(str(mix))
        total_s += info.duration
        n_ok    += 1
    except Exception as e:
        print(f'  WARN: could not read {mix}: {e}')

h    = int(total_s) // 3600
mins = (int(total_s) % 3600) // 60
hours = total_s / 3600

print(f'Curated tracks    : {len(CURATED_TRACKS)}')
print(f'Tracks with audio : {n_ok}')
if missing:
    print(f'Missing mix.flac  : {len(missing)}  (first 5: {missing[:5]})')
print(f'Total duration    : {h}h {mins}m  ({hours:.2f} h)')
print(f'Est. segments     : ~{n_ok * 36} × 5 s  (at 3 min/track avg)')
print()

if hours >= MIN_HOURS:
    print(f'✓ DURATION GATE PASSED: {hours:.2f} h >= {MIN_HOURS} h')
    print('  → Proceed to Cell 2 (slakh_adapter on curated tracks).')
else:
    deficit_min = (MIN_HOURS - hours) * 60
    print(f'✗ DURATION GATE FAILED: {hours:.2f} h < {MIN_HOURS} h (need {deficit_min:.0f} more minutes)')
    print('  Options:')
    print('  1. Re-run Cell 1.9, flip some Reject → Keep, click Save + Finalize.')
    print('  2. Lower MIN_HOURS in this cell if your Israeli dataset is also < 3 h.')


<!-- §37-stub -->
## Cell 9 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 1.98 — MFCC similarity filter
# ============================================================
# Runs on raw mix.flac — no tensors needed, so runs before Cell 2.
#
# Two checks:
#   1. Near-duplicate detection   (cosine sim > DUP_THRESHOLD between any pair)
#      → likely the same song appearing twice. One track is removed.
#   2. Outlier detection          (cosine sim to set centroid < OUTLIER_THRESHOLD)
#      → stylistically doesn't belong in the set.
#
# ⚠️  SKIP_FOR_SYNTHETIC=True (default) for Slakh2100 and similar GM-rendered
#     datasets. Slakh uses identical GM soundfont timbres for every track, so
#     MFCC sees ~0.95 cosine similarity everywhere — not because the songs are
#     duplicates, but because they all sound like "General MIDI synth". This
#     makes the near-duplicate filter useless (it removes everything).
#     The real diversity in Slakh is in rhythm, note density, and mix ratios,
#     which MFCC does not capture. For Slakh, the hearing test + BPM filter
#     already did the quality filtering; no further filter is needed here.
#
# Set SKIP_FOR_SYNTHETIC=False only for datasets with real/varied instrument
# recordings (e.g. the Israeli YouTube training data).
#
# CURATED_TRACKS is updated in-memory. slakh_curated.txt is NOT rewritten
# unless you set WRITE_FILTERED=True.
#
# Runtime: ~2–4 min for 254 tracks (30s MFCC extraction per track).
# ============================================================

import pathlib, numpy as np, soundfile as sf, librosa

SKIP_FOR_SYNTHETIC = True   # ← set False for real-instrument datasets
DUP_THRESHOLD      = 0.92   # cosine sim above this → near-duplicate
OUTLIER_THRESHOLD  = 0.30   # cosine sim to centroid below this → outlier
CLIP_START_S       = 30     # skip intro
CLIP_DURATION_S    = 30     # seconds of audio to analyse
WRITE_FILTERED     = False  # set True to overwrite slakh_curated.txt with filtered list

if SKIP_FOR_SYNTHETIC:
    print('SKIP_FOR_SYNTHETIC=True — MFCC filter skipped.')
    print()
    print('Reason: Slakh2100 is rendered from a single GM soundfont, so every')
    print('track shares the same synthesized timbres. MFCC captures timbre, not')
    print('rhythm or note content, so pairwise similarity is ~0.92–0.99 for ALL')
    print('track pairs — not because they are duplicates, but because they are')
    print('all "General MIDI synth". Running the filter here would incorrectly')
    print('remove 250+ tracks from a 254-track set.')
    print()
    print(f'CURATED_TRACKS unchanged → {len(CURATED_TRACKS)} tracks.')
    print('→ Proceed to Cell 2.')
else:
    # ── 1. Extract mean MFCC vector (40 coeffs) for each track ───────────────
    print(f'Extracting MFCCs for {len(CURATED_TRACKS)} tracks ({CLIP_DURATION_S}s clips)...')
    vecs   = {}   # tid -> unit-norm MFCC vector
    failed = []

    for i, tid in enumerate(CURATED_TRACKS):
        mix = pathlib.Path(SLAKH_RAW) / tid / 'mix.flac'
        try:
            info   = sf.info(str(mix))
            start  = int(info.samplerate * CLIP_START_S)
            frames = int(info.samplerate * CLIP_DURATION_S)
            data, sr = sf.read(str(mix), start=start, frames=frames,
                               dtype='float32', always_2d=False)
            if data.ndim > 1:
                data = data.mean(axis=1)
            if sr != 22050:
                data = librosa.resample(data, orig_sr=sr, target_sr=22050)
            mfcc = librosa.feature.mfcc(y=data, sr=22050, n_mfcc=40)
            vec  = mfcc.mean(axis=1)          # shape (40,)
            norm = np.linalg.norm(vec)
            vecs[tid] = vec / norm if norm > 0 else vec
        except Exception as e:
            print(f'  WARN {tid}: {e}')
            failed.append(tid)

        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(CURATED_TRACKS)} done')

    print(f'Extracted: {len(vecs)}  |  Failed: {len(failed)}')

    tids   = list(vecs.keys())
    matrix = np.stack([vecs[t] for t in tids])   # (N, 40)

    # ── 2. Near-duplicate detection ──────────────────────────────────────────
    print(f'\nNear-duplicate scan (threshold={DUP_THRESHOLD}) ...')
    sim_matrix = matrix @ matrix.T   # (N, N) cosine similarities
    np.fill_diagonal(sim_matrix, 0)

    removed_dup = set()
    dup_pairs   = []
    for i in range(len(tids)):
        if tids[i] in removed_dup:
            continue
        for j in range(i + 1, len(tids)):
            if tids[j] in removed_dup:
                continue
            if sim_matrix[i, j] >= DUP_THRESHOLD:
                removed_dup.add(tids[j])   # keep i, remove j
                dup_pairs.append((tids[i], tids[j], float(sim_matrix[i, j])))

    if dup_pairs:
        print(f'  Near-duplicate pairs found: {len(dup_pairs)}')
        for a, b, s in dup_pairs:
            print(f'    {a} ↔ {b}  sim={s:.3f}  → removed {b}')
    else:
        print('  No near-duplicates found.')

    # ── 3. Outlier detection ──────────────────────────────────────────────────
    print(f'\nOutlier scan (centroid threshold={OUTLIER_THRESHOLD}) ...')
    centroid   = matrix.mean(axis=0)
    centroid  /= np.linalg.norm(centroid)
    cent_sims  = matrix @ centroid   # (N,) similarity to centroid

    removed_out = set()
    for i, tid in enumerate(tids):
        if tid in removed_dup:
            continue
        if cent_sims[i] < OUTLIER_THRESHOLD:
            removed_out.add(tid)
            print(f'  Outlier: {tid}  sim_to_centroid={cent_sims[i]:.3f}')

    if not removed_out:
        print('  No outliers found.')

    # ── 4. Summary & update CURATED_TRACKS ───────────────────────────────────
    all_removed = removed_dup | removed_out
    filtered    = [t for t in CURATED_TRACKS if t not in all_removed]

    print(f'\n── Summary ─────────────────────────────────────────')
    print(f'  Original  : {len(CURATED_TRACKS)} tracks')
    print(f'  Duplicates removed : {len(removed_dup)}')
    print(f'  Outliers removed   : {len(removed_out)}')
    print(f'  Remaining : {len(filtered)} tracks')

    # Similarity stats for the filtered set
    filt_idx  = [i for i, t in enumerate(tids) if t not in all_removed]
    filt_mat  = matrix[filt_idx]
    filt_sims = (filt_mat @ filt_mat.T)
    np.fill_diagonal(filt_sims, np.nan)
    print(f'  Mean pairwise sim  : {np.nanmean(filt_sims):.3f}  '
          f'(higher = more stylistically consistent)')
    print(f'  Min centroid sim   : {cent_sims[filt_idx].min():.3f}')

    CURATED_TRACKS = filtered
    print(f'\nCURATED_TRACKS updated in memory → {len(CURATED_TRACKS)} tracks.')

    if WRITE_FILTERED:
        CURATED_FILE = str(pathlib.Path(DRIVE_ROOT) / 'slakh_curated.txt')
        pathlib.Path(CURATED_FILE).write_text('\n'.join(CURATED_TRACKS))
        print(f'Wrote filtered list → {CURATED_FILE}')
    else:
        print('WRITE_FILTERED=False → slakh_curated.txt unchanged on disk.')
    print('→ Proceed to Cell 2.')


<!-- §37-stub -->
## Cell 10 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 1.99 — Duration-budget sampler
# ============================================================
# Subsamples CURATED_TRACKS to a target audio duration so you can
# run quick experiments on less data and scale up incrementally.
#
#   TARGET_HOURS = 3.0  → keep enough tracks to cover ~3h
#   TARGET_HOURS = 6.0  → keep enough tracks to cover ~6h
#   TARGET_HOURS = None → use all curated tracks (no subsampling)
#
# Tracks are shuffled (seeded) before selection so the sample is
# representative rather than just the first N alphabetically.
# Durations are read from file headers only — takes ~30 seconds.
# ============================================================

import pathlib, random, soundfile as sf

TARGET_HOURS = 3.0   # ← change to 6.0, 12.0, or None to scale up
RANDOM_SEED  = 42

if TARGET_HOURS is None:
    print(f'TARGET_HOURS=None → using all {len(CURATED_TRACKS)} curated tracks.')
    print('→ Proceed to Cell 2.')
else:
    TARGET_SECS = TARGET_HOURS * 3600

    tracks_shuffled = list(CURATED_TRACKS)
    random.seed(RANDOM_SEED)
    random.shuffle(tracks_shuffled)

    print(f'Reading durations for {len(tracks_shuffled)} tracks (header-only)...')
    selected   = []
    cumulative = 0.0
    for tid in tracks_shuffled:
        mix = pathlib.Path(SLAKH_RAW) / tid / 'mix.flac'
        try:
            info = sf.info(str(mix))
            dur  = info.frames / info.samplerate
        except Exception:
            dur = 0.0
        selected.append((tid, dur))
        cumulative += dur
        if cumulative >= TARGET_SECS:
            break

    CURATED_TRACKS = [t for t, _ in selected]
    actual_h = cumulative / 3600

    print(f'\n── Duration sampler ─────────────────────────────────')
    print(f'  Target   : {TARGET_HOURS:.1f}h')
    print(f'  Selected : {len(CURATED_TRACKS)} tracks  ({actual_h:.2f}h actual)')
    print(f'  Skipped  : {len(tracks_shuffled) - len(CURATED_TRACKS)} tracks  '
          f'(raise TARGET_HOURS to include more)')
    print(f'  Seed     : {RANDOM_SEED}')
    print(f'\nCURATED_TRACKS updated → {len(CURATED_TRACKS)} tracks.')
    print('→ Proceed to Cell 2.')


<!-- §37-stub -->
## Cell 11 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
import subprocess, pathlib, tempfile, os

# CURATED_TRACKS is set by Cell 1.95.
# slakh_adapter.py receives a --track_ids file (one ID per line) to process only curated tracks.
assert 'CURATED_TRACKS' in dir() and CURATED_TRACKS, (
    'CURATED_TRACKS not set. Run Cell 1.95 first.'
)

print(f'Processing {len(CURATED_TRACKS)} curated tracks ...')

# Write track IDs to a temp file so slakh_adapter.py can filter
track_ids_file = '/tmp/slakh_curated_ids.txt'
with open(track_ids_file, 'w') as f:
    f.write('\n'.join(CURATED_TRACKS))

cmd = [
    'python', 'preprocessing/slakh_adapter.py',
    '--slakh_split_dir', SLAKH_RAW,
    '--out_root',        SLAKH_PROCESSED,
    '--manifest_out',    SLAKH_MANIFEST,
    '--track_ids_file',  track_ids_file,
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
assert result.returncode == 0, f'slakh_adapter.py failed with code {result.returncode}'

import csv
with open(SLAKH_MANIFEST, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print(f'Slakh manifest: {len(rows)} segments from {len({r["song_id"] for r in rows})} tracks')


<!-- §37-stub -->
## Cell 12 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 3 — Split manifest into train/val/test
# ============================================================

import subprocess

SPLIT_DIR = f'{MANIFEST_DIR}/slakh_splits'
import pathlib; pathlib.Path(SPLIT_DIR).mkdir(parents=True, exist_ok=True)

cmd = [
    'python', 'preprocessing/split_dataset.py',
    '--manifest', SLAKH_MANIFEST,
    '--out_dir',  SPLIT_DIR,
    '--train', '0.8',
    '--val',   '0.1',
    '--test',  '0.1',
    '--seed',  '42',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
assert result.returncode == 0, 'split_dataset.py failed'

import csv
def count_csv(p):
    with open(p, newline='', encoding='utf-8') as f:
        return len(list(csv.DictReader(f)))

TRAIN_CSV = f'{SPLIT_DIR}/train.csv'
VAL_CSV   = f'{SPLIT_DIR}/val.csv'
TEST_CSV  = f'{SPLIT_DIR}/test.csv'
print(f'train={count_csv(TRAIN_CSV)} | val={count_csv(VAL_CSV)} | test={count_csv(TEST_CSV)}')

<!-- §37-stub -->
## Cell 13 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 3.1 — Tensor health check (shape, dtype, size, contiguous)
# ============================================================
# WHY THIS MATTERS:
#   slakh_adapter slices numpy arrays non-contiguously:
#     mel_seg = torch.from_numpy(log_mel[:, s:e])
#   A column slice of a C-contiguous [80, T] array is a VIEW.
#   torch.from_numpy() wraps the full parent storage → each .pt
#   file stores the ENTIRE track's data, not just 430 frames.
#
#   Effect:  expected ~1.2 GB dataset → actual ~56 GB
#            expected ~134 KB/file    → actual ~6–20 MB/file
#
# If "CONTIGUOUS: NO" is printed below, run Cell 3.2 to fix.
# ============================================================

import torch, pathlib, csv, random

# ── 1. Sample a few mel + piano_roll files ──────────────────
with open(TRAIN_CSV, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
sample_rows = random.sample(rows, min(5, len(rows)))

print(f'{"FILE":<40} {"SHAPE":<20} {"CONT?":<6} '
      f'{"STORAGE MB":>10} {"ACTUAL MB":>10} {"FILE MB":>8}')
print('-' * 100)

issues = []
for row in sample_rows:
    for key, label in [('segment_path', 'mel'), ('score_path', 'pr')]:
        fpath = pathlib.Path(SLAKH_PROCESSED) / row[key]
        t = torch.load(fpath, weights_only=True)
        storage_mb    = t.storage().nbytes() / 1e6
        contiguous_mb = t.numel() * t.element_size() / 1e6
        file_mb       = fpath.stat().st_size / 1e6
        is_cont       = t.is_contiguous()
        flag = '' if is_cont else '  ← BUG: full track stored'
        print(f'{label+"/"+fpath.name:<40} {str(tuple(t.shape)):<20} '
              f'{"YES":<6} '
              f'{storage_mb:>10.2f} {contiguous_mb:>10.3f} {file_mb:>8.2f}{flag}')
        if not is_cont:
            issues.append(fpath)

# ── 2. Dataset-level size estimate ──────────────────────────
pt_files = list(pathlib.Path(SLAKH_PROCESSED).rglob('*.pt'))
n_files  = len(pt_files)
total_gb = sum(f.stat().st_size for f in pt_files) / 1e9
n_segs   = n_files // 2
exp_kb_per_seg = (80 * 430 * 4 + 2 * 128 * 430 * 4) / 1024
exp_gb = n_segs * exp_kb_per_seg / 1024 / 1024

print()
print(f'Files:            {n_files}  ({n_segs} segments × 2)')
print(f'Actual disk:      {total_gb:.1f} GB')
print(f'Expected (fixed): {exp_gb:.2f} GB  ({exp_kb_per_seg:.0f} KB/segment)')
print(f'Bloat factor:     {total_gb/max(exp_gb,0.001):.0f}×')

batch = 32
avg_mb_file = (total_gb * 1e3) / max(n_files, 1)
exp_mb_file = exp_kb_per_seg / 2 / 1024  # avg per file (mel+pr)/2
print()
print(f'I/O per training step (batch={batch}):')
print(f'  Current:  {avg_mb_file*batch:.0f} MB read  (~{avg_mb_file*batch/50:.1f}s @ 50 MB/s + FUSE)')
print(f'  If fixed: {exp_mb_file*batch:.0f} MB read  (~{exp_mb_file*batch/50:.2f}s @ 50 MB/s + FUSE)')

# ── 3. Verdict ───────────────────────────────────────────────
print()
if issues:
    print(f'⚠  TENSORS ARE NON-CONTIGUOUS ({len(issues)} of {len(issues)} sampled)')
    print(f'   → Run Cell 3.2 to re-save all files as contiguous (local, fast).')
    print(f'   → After fix: ~{exp_gb:.2f} GB dataset, training may not need /tmp cache at all.')
else:
    print('✓  All sampled tensors are contiguous — proceed to Cell 3.5.')


<!-- §37-stub -->
## Cell 14 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 3.2 — Fix existing tensors: re-save as contiguous
# ============================================================
# Overwrites each .pt file in slakh_processed/ with its
# contiguous equivalent.  Values are IDENTICAL — only the
# on-disk storage layout changes.
#
# Result: ~56 GB → ~1.2 GB (48× smaller)
#
# ── HOW TO RUN ──────────────────────────────────────────────
# OPTION A — LOCAL (recommended, ~2 min):
#   1. slakh_processed is already offline in Google Drive for Desktop.
#   2. Run this Python script in a LOCAL terminal (not Colab):
#
#      python - << 'EOF'
#      import torch, pathlib
#      from concurrent.futures import ThreadPoolExecutor, as_completed
#      src = pathlib.Path(r"G:\My Drive\MusicProject\slakh_processed")
#      files = list(src.rglob("*.pt"))
#      print(f"Found {len(files)} .pt files ({sum(f.stat().st_size for f in files)/1e9:.1f} GB)")
#      def fix(f):
#          t = torch.load(f, weights_only=True)
#          if not t.is_contiguous():
#              torch.save(t.contiguous(), f)
#              return True
#          return False
#      fixed = 0
#      with ThreadPoolExecutor(max_workers=8) as ex:
#          futs = {ex.submit(fix, f): f for f in files}
#          for i, fut in enumerate(as_completed(futs), 1):
#              if fut.result(): fixed += 1
#              if i % 500 == 0: print(f"  {i}/{len(files)} done, {fixed} fixed so far")
#      print(f"Done. {fixed}/{len(files)} files re-saved.")
#      EOF
#
#   3. Wait for Google Drive for Desktop to upload the smaller files
#      (~1.2 GB upload vs 55.8 GB already on Drive — takes ~2 min).
#   4. In Drive for Desktop: right-click slakh_processed → "Free up space"
#      → removes the local offline cache (frees ~54 GB on your laptop).
#
# OPTION B — IN COLAB (~25 min via Drive FUSE, parallel):
#   Uncomment the block below and run this cell in Colab.
# ============================================================

# ── OPTION B: Colab re-save (uncomment to run) ──────────────
# import torch, pathlib, time
# from concurrent.futures import ThreadPoolExecutor, as_completed
#
# src = pathlib.Path(SLAKH_PROCESSED)
# files = list(src.rglob('*.pt'))
# before_gb = sum(f.stat().st_size for f in files) / 1e9
# print(f'Found {len(files)} files  ({before_gb:.1f} GB)  — re-saving as contiguous...')
#
# def fix_one(f):
#     t = torch.load(f, weights_only=True)
#     if not t.is_contiguous():
#         torch.save(t.contiguous(), f)
#         return True
#     return False
#
# t0 = time.time()
# fixed = 0
# with ThreadPoolExecutor(max_workers=32) as ex:
#     futs = {ex.submit(fix_one, f): f for f in files}
#     for i, fut in enumerate(as_completed(futs), 1):
#         if fut.result(): fixed += 1
#         if i % 500 == 0 or i == len(files):
#             elapsed = time.time() - t0
#             print(f'  {i}/{len(files)}  ({elapsed/60:.1f} min)  {fixed} fixed')
#
# after_gb = sum(f.stat().st_size for f in files) / 1e9
# print(f'\nBefore: {before_gb:.1f} GB   After: {after_gb:.2f} GB   '
#       f'Saved: {before_gb - after_gb:.1f} GB  ({time.time()-t0:.0f}s)')

print('Cell 3.2: see comments above for re-save instructions.')
print('Run OPTION A locally for fastest results (~2 min).')
print('After fix + Drive sync: turn off "Available offline" in Drive for Desktop')
print('  → frees ~54 GB of local disk space.')


<!-- §37-stub -->
## Cell 15 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 3.5 — Cache processed data to local /tmp (I/O speedup)
# ============================================================
# PROBLEM: Reading .pt files directly from Drive during training
#   causes ~8s/step (FUSE per-file overhead × 4300 files).
#
# FIX: Copy slakh_processed/ to /tmp (Colab local SSD) once.
#   Before: ~8s/step  (~69h for 30k steps)
#   After:  ~1s/step  (~8h  for 30k steps)
#
# SPEED: parallel copy with 32 threads overlaps FUSE per-file
#   overhead instead of paying it sequentially.
#   Sequential (shutil.copytree): ~25 min
#   Parallel   (ThreadPoolExecutor, 32 workers): ~8-12 min
#
# Re-run each session (/tmp is wiped on disconnect).
# ============================================================

import shutil, pathlib, time
from concurrent.futures import ThreadPoolExecutor, as_completed

SLAKH_PROCESSED_LOCAL = '/tmp/slakh_processed'
N_WORKERS = 32          # parallel Drive→/tmp copy threads

src = pathlib.Path(SLAKH_PROCESSED)
dst = pathlib.Path(SLAKH_PROCESSED_LOCAL)

if dst.exists() and any(dst.rglob('*.pt')):
    n = sum(1 for _ in dst.rglob('*.pt'))
    print(f'Already cached at {dst}  ({n} .pt files) — skipping.')
else:
    if dst.exists():
        shutil.rmtree(str(dst))
    dst.mkdir(parents=True, exist_ok=True)

    # Collect all files (not just .pt — keeps directory structure intact)
    all_files = [f for f in src.rglob('*') if f.is_file()]
    size_gb = sum(f.stat().st_size for f in all_files) / 1e9
    print(f'Copying {len(all_files)} files ({size_gb:.0f} GB) '
          f'with {N_WORKERS} parallel threads ...')
    print('Expected time: ~8-12 min')

    def _copy_one(src_f):
        dst_f = dst / src_f.relative_to(src)
        dst_f.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(src_f), str(dst_f))

    t0 = time.time()
    done = 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = {ex.submit(_copy_one, f): f for f in all_files}
        for fut in as_completed(futures):
            fut.result()          # raise immediately if a copy failed
            done += 1
            if done % 500 == 0 or done == len(all_files):
                elapsed = time.time() - t0
                rate    = done / elapsed
                eta_s   = (len(all_files) - done) / rate if rate > 0 else 0
                print(f'  {done}/{len(all_files)} files  '
                      f'({elapsed/60:.1f} min elapsed, '
                      f'ETA {eta_s/60:.1f} min)')

    elapsed = time.time() - t0
    n = sum(1 for _ in dst.rglob('*.pt'))
    print(f'\nDone in {elapsed/60:.1f} min — {n} .pt files cached.')

print(f'\nSLAKH_PROCESSED_LOCAL = {SLAKH_PROCESSED_LOCAL}')
print('Cell 4 will use this path as --data_root.')


<!-- §37-stub -->
## Cell 16 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 4 — Training (MAX_STEPS steps, run overnight or across sessions)
# Re-run this cell to resume automatically via --resume_from auto
#
# Uses SLAKH_PROCESSED_LOCAL (/tmp/slakh_processed) set by Cell 3.5.
# If Cell 3.5 was skipped, falls back to the Drive path.
# ============================================================

import sys

_data_root = locals().get('SLAKH_PROCESSED_LOCAL') or SLAKH_PROCESSED
print(f'data_root = {_data_root}')
if 'tmp' in _data_root:
    print('  ✓ Using local /tmp cache — expect ~1s/step')
else:
    print('  ⚠ Using Drive path — expect ~8s/step  (run Cell 3.5 first!)')

cmd = [
    'python', 'train.py',
    '--config',          'configs/default.yaml',
    '--train_manifest',  TRAIN_CSV,
    '--val_manifest',    VAL_CSV,
    '--data_root',       _data_root,
    '--max_steps',       str(MAX_STEPS),
    '--ckpt_dir',        CKPT_DIR,
    '--log_dir',         LOG_DIR,
    '--resume_from',     'auto',
]
print('\nCommand:\n  ' + ' '.join(cmd))
print()
print('Starting training... (leave this cell running overnight)')
print('TensorBoard logs will be written to:', LOG_DIR)
print('Checkpoints will be saved to:        ', CKPT_DIR)
print()

!python train.py \
    --config configs/default.yaml \
    --train_manifest {TRAIN_CSV} \
    --val_manifest {VAL_CSV} \
    --data_root {_data_root} \
    --max_steps {MAX_STEPS} \
    --ckpt_dir {CKPT_DIR} \
    --log_dir {LOG_DIR} \
    --resume_from auto


<!-- §37-stub -->
## Cell 17 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 5 — Plot train/val loss curves (matplotlib, no TensorBoard UI)
# Run after training (or mid-training) to visualise curves.
#
# Train and val loss are shown on the same chart for direct comparison
# (generalization gap visible at a glance). LR schedule on a second panel.
# Reads TensorBoard event files written by train.py via SummaryWriter.
# ============================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pathlib

# Cell 1 must have run to set LOG_DIR, CKPT_DIR
assert 'LOG_DIR' in dir(), 'LOG_DIR not set — re-run Cell 1 first.'

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tensorboard'])
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

print(f'Reading event files from: {LOG_DIR}')
ea = EventAccumulator(LOG_DIR)
ea.Reload()
available = ea.Tags().get('scalars', [])
print(f'Available scalars: {available}')

def get_scalar(tag):
    if tag not in available:
        return [], []
    events = ea.Scalars(tag)
    return [e.step for e in events], [e.value for e in events]

train_steps, train_loss = get_scalar('train/loss')
val_steps,   val_loss   = get_scalar('val/loss')
lr_steps,    lr_vals    = get_scalar('train/lr')

# ── Print key statistics ────────────────────────────────────
best_val, best_step = None, None
if val_loss:
    best_idx   = val_loss.index(min(val_loss))
    final_val  = val_loss[-1]
    best_val   = val_loss[best_idx]
    best_step  = val_steps[best_idx]
    print(f'\nVal loss summary:')
    print(f'  Best  : {best_val:.4f}  at step {best_step}')
    print(f'  Final : {final_val:.4f} at step {val_steps[-1]}')
    print(f'  Gate  : [0.15, 0.30]  →  {"PASSED ✓" if 0.15 <= best_val <= 0.30 else "FAILED ✗"}')
else:
    print('WARNING: no val/loss events found.')

# ── Plot: 2 panels — (1) train + val overlaid, (2) LR schedule ──────────────
fig, (ax_loss, ax_lr) = plt.subplots(1, 2, figsize=(14, 4))

if train_loss:
    ax_loss.plot(train_steps, train_loss, lw=0.8, alpha=0.55, color='steelblue',
                 label='train/loss')
if val_loss:
    ax_loss.plot(val_steps, val_loss, marker='o', ms=3, color='tomato',
                 label='val/loss')
    ax_loss.axhline(best_val, color='green', ls='--', lw=0.8,
                    label=f'best val={best_val:.4f} @ step {best_step}')
    ax_loss.axhspan(0.15, 0.30, alpha=0.08, color='green', label='gate [0.15, 0.30]')
ax_loss.set_xlabel('step')
ax_loss.set_ylabel('loss')
ax_loss.set_title('Train vs. Validation loss')
ax_loss.legend(fontsize=8)

if lr_vals:
    ax_lr.plot(lr_steps, lr_vals, lw=0.8, color='orange')
ax_lr.set_xlabel('step')
ax_lr.set_ylabel('lr')
ax_lr.set_title('Learning rate schedule')

plt.tight_layout()
fig_path = pathlib.Path(CKPT_DIR) / 'loss_curves.png'
plt.savefig(str(fig_path), dpi=120)
plt.show()
print(f'Saved: {fig_path}')


<!-- §37-stub -->
## Cell 18 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 5.5 — Mel progression gallery
# Shows how generated mels evolve during training.
# Each image (3 panels): Target | Generated | Difference
#
# PRIMARY path: reads PNGs saved by train.py during training
#   (written to {CKPT_DIR}/mel_viz/ every 2 checkpoints)
# FALLBACK path: if no PNGs exist, generates them retroactively
#   by running DDIM on every checkpoint currently on Drive.
#   Useful when training already completed before this cell existed.
# ============================================================

import csv, pathlib, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

assert 'CKPT_DIR' in dir(), 'CKPT_DIR not set — re-run Cell 1 first.'
sys.path.insert(0, '/content/MusicProject')

viz_dir = pathlib.Path(CKPT_DIR) / 'mel_viz'
viz_dir.mkdir(exist_ok=True)

# ── FALLBACK: generate PNGs from saved checkpoints ──────────────────────────
existing_pngs = sorted(viz_dir.glob('step_*.png'))
if not existing_pngs:
    print('No mel_viz PNGs found — generating retroactively from saved checkpoints...')

    from omegaconf import OmegaConf
    from model.unet import UNet1D
    from model.diffusion import GaussianDiffusion

    cfg    = OmegaConf.load('configs/default.yaml')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    mc, cc, sc, dc, tc = cfg.model, cfg.conditioning, cfg.sampling, cfg.diffusion, cfg.training

    # Load one fixed val segment (same across all checkpoints for fair comparison)
    with open(VAL_CSV, newline='', encoding='utf-8') as f:
        row = list(csv.DictReader(f))[0]
    tgt_mel  = torch.load(pathlib.Path(SLAKH_PROCESSED) / row['segment_path'], weights_only=True)
    score_raw = torch.load(pathlib.Path(SLAKH_PROCESSED) / row['score_path'],   weights_only=True)
    print(f'Using val segment: song_id={row["song_id"]}  segment={row["segment_idx"]}')

    # Collect all step checkpoints + best_val, sorted by step number
    step_ckpts = sorted(
        pathlib.Path(CKPT_DIR).glob('step_*.pt'),
        key=lambda p: int(p.stem.split('_')[1]),
    )
    best_ckpt = pathlib.Path(CKPT_DIR) / 'best_val.pt'
    all_ckpts = step_ckpts + ([best_ckpt] if best_ckpt.exists() else [])
    print(f'Found {len(all_ckpts)} checkpoint(s): {[p.name for p in all_ckpts]}')

    for ckpt_path in all_ckpts:
        step_tag = ckpt_path.stem          # e.g. 'step_029000' or 'best_val'
        png_path = viz_dir / f'{step_tag}.png'
        if png_path.exists():
            print(f'  {step_tag}: already exists, skipping.')
            continue

        print(f'  {step_tag}: loading checkpoint…')
        state = torch.load(ckpt_path, map_location=device, weights_only=False)

        model = UNet1D(
            mel_channels=mc.mel_channels,
            score_channels=mc.score_channels,
            base_channels=mc.base_channels,
            channel_mults=list(mc.channel_mults),
            num_res_blocks_enc=mc.num_res_blocks_enc,
            num_res_blocks_dec=mc.num_res_blocks_dec,
            attention_levels=list(mc.attention_levels),
            attn_heads=mc.attention_heads,
            n_groups=mc.n_groups,
            dropout=mc.dropout,
            n_versions=cc.n_versions,
            version_emb_dim=cc.version_emb_dim,
            time_emb_dim=cc.time_emb_dim,
        ).to(device).eval()

        weights_key = 'ema' if 'ema' in state else ('model' if 'model' in state else None)
        assert weights_key, f'Unknown checkpoint format: keys={list(state.keys())}'
        model.load_state_dict(state[weights_key], strict=True)

        diffusion = GaussianDiffusion(
            model=model,
            T=dc.T_train,
            n_versions=cc.n_versions,
            cfg_score=sc.cfg_score,
            cfg_version=sc.cfg_version,
            cfg_drop_score=tc.cfg_drop_score,
            cfg_drop_version=tc.cfg_drop_version,
            cfg_drop_both=tc.cfg_drop_both,
        ).to(device).eval()

        score_t = score_raw.unsqueeze(0).to(device)
        score_t = score_t.view(1, -1, score_t.shape[-1])   # [1, 256, 430]
        vid     = torch.zeros(1, dtype=torch.long, device=device)

        print(f'  {step_tag}: running DDIM ({sc.N_ddim} steps)…')
        with torch.no_grad():
            gen_mel = diffusion.ddim_sample(score=score_t, version_id=vid, N=sc.N_ddim)
        gen_np = gen_mel.squeeze(0).cpu().numpy()
        tgt_np = tgt_mel.numpy()

        fig, axes = plt.subplots(1, 3, figsize=(15, 3))
        axes[0].imshow(tgt_np, origin='lower', aspect='auto', cmap='magma')
        axes[0].set_title('Target mel')
        axes[1].imshow(gen_np, origin='lower', aspect='auto', cmap='magma')
        axes[1].set_title('Generated mel')
        diff = gen_np - tgt_np
        axes[2].imshow(diff, origin='lower', aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
        axes[2].set_title('Difference (gen − target)')
        for ax in axes:
            ax.axis('off')
        plt.suptitle(f'Checkpoint: {step_tag}', fontsize=11)
        plt.tight_layout()
        plt.savefig(str(png_path), dpi=100, bbox_inches='tight')
        plt.close()
        print(f'  {step_tag}: saved {png_path.name}')

        # Free GPU memory before next checkpoint
        del model, diffusion
        torch.cuda.empty_cache()

    print('Retroactive generation done.')

# ── DISPLAY gallery ──────────────────────────────────────────────────────────
pngs = sorted(viz_dir.glob('*.png'))
n = len(pngs)
if n == 0:
    print('No mel PNGs to display.')
else:
    print(f'Displaying {n} mel snapshot(s)…')
    fig, axes = plt.subplots(n, 1, figsize=(15, 3.2 * n))
    if n == 1:
        axes = [axes]
    for ax, png in zip(axes, pngs):
        img = mpimg.imread(str(png))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(png.stem.replace('_', ' '), fontsize=9, pad=2)
    plt.suptitle('Mel Gallery — Target | Generated | Difference', fontsize=12, y=1.0)
    plt.tight_layout()
    gallery_path = pathlib.Path(CKPT_DIR) / 'mel_gallery.png'
    plt.savefig(str(gallery_path), dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Gallery saved: {gallery_path}')


<!-- §37-stub -->
## Cell 19 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 5.6 — Training loss curves (train + val)
# ============================================================
# Reads loss_log.csv written by train.py alongside checkpoints.
# Run any time during or after training to see progress.

import pathlib, csv
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

loss_csv = pathlib.Path(CKPT_DIR) / 'loss_log.csv'
assert loss_csv.exists(), (
    f'loss_log.csv not found at {loss_csv}.\n'
    'Make sure you are running train.py from commit d88c3ac+ and have started training.'
)

train_steps, train_losses = [], []
val_steps,   val_losses   = [], []

with open(loss_csv, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        s = int(row['step'])
        if row['train_loss']:
            train_steps.append(s)
            train_losses.append(float(row['train_loss']))
        if row['val_loss']:
            val_steps.append(s)
            val_losses.append(float(row['val_loss']))

print(f'Train rows: {len(train_steps)}  |  Val rows: {len(val_steps)}')
if train_steps:
    print(f'Steps: {train_steps[0]} → {train_steps[-1]}')
if val_losses:
    print(f'Best val loss: {min(val_losses):.4f} at step {val_steps[val_losses.index(min(val_losses))]}')

fig, ax = plt.subplots(figsize=(11, 4))

if train_losses:
    ax.plot(train_steps, train_losses, color='steelblue', lw=1.0, alpha=0.7, label='train loss')
    # Smoothed trend (rolling mean over 20 points)
    if len(train_losses) >= 20:
        import numpy as np
        kernel = np.ones(20) / 20
        smooth = np.convolve(train_losses, kernel, mode='valid')
        ax.plot(train_steps[19:], smooth, color='steelblue', lw=2.0, label='train (smoothed)')

if val_losses:
    ax.plot(val_steps, val_losses, 'o-', color='tomato', lw=1.5, ms=5, label='val loss')
    best_idx = val_losses.index(min(val_losses))
    ax.axvline(val_steps[best_idx], color='tomato', ls='--', lw=0.8, alpha=0.6)
    ax.annotate(f'best {min(val_losses):.4f}',
                xy=(val_steps[best_idx], min(val_losses)),
                xytext=(8, 8), textcoords='offset points',
                fontsize=8, color='tomato')

ax.axhspan(0.15, 0.30, color='green', alpha=0.08, label='val gate [0.15, 0.30]')
ax.set_xlabel('step')
ax.set_ylabel('L1 diffusion loss')
ax.set_title('Slakh sanity training — loss curves')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))
ax.grid(True, lw=0.3)
plt.tight_layout()
plt.show()


<!-- §37-stub -->
## Cell 20 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 5.7 — Post-training batch audio (BigVGAN all checkpoints)
# ============================================================
# Vocodes every step_*_mels.pt in CKPT_DIR/samples/ → audio/
# Produces [A] raw and [B] contrast-stretched .wav per checkpoint.
# Run after training completes (or at any point during training).

import pathlib, torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

sample_dir = pathlib.Path(CKPT_DIR) / 'samples'
audio_dir  = pathlib.Path(CKPT_DIR) / 'audio'
audio_dir.mkdir(exist_ok=True)

mel_files = sorted(sample_dir.glob('step_*_mels.pt')) if sample_dir.exists() else []

if not mel_files:
    print('No step_*_mels.pt files found — run training first.')
else:
    from postprocessing.bigvgan_vocoder import BigVGANVocoder
    voc = BigVGANVocoder(model_name='bigvgan_22k',
                         device='cuda' if torch.cuda.is_available() else 'cpu')

    def _norm_to_logmag(mel_norm_np):
        mel_db = (mel_norm_np + 1.0) / 2.0 * 80.0 - 80.0
        return (mel_db * (np.log(10) / 20.0)).astype(np.float32)

    for mel_path in mel_files:
        step_tag = mel_path.stem.replace('step_', '').replace('_mels', '')
        tgt_path = sample_dir / f'step_{step_tag}_targets.pt'

        mels = torch.load(mel_path, weights_only=True)          # [B, 80, 430]
        tgt  = torch.load(tgt_path, weights_only=True) if tgt_path.exists() else None

        for i in range(mels.shape[0]):
            gen_np = mels[i].numpy()

            # A. Raw generated audio
            audio_raw = voc.mel_to_audio(
                torch.from_numpy(_norm_to_logmag(gen_np)).unsqueeze(0))
            raw_wav = audio_dir / f'step_{step_tag}_b{i}_raw.wav'
            sf.write(str(raw_wav), audio_raw, voc.h.sampling_rate)

            # B. Contrast-stretched (match target mean/std)
            if tgt is not None:
                tgt_np  = tgt[i].numpy()
                t_mean, t_std = float(tgt_np.mean()), float(tgt_np.std())
                g_mean, g_std = float(gen_np.mean()), float(gen_np.std())
                stretched = (gen_np - g_mean) / (g_std + 1e-8) * t_std + t_mean
                stretched = stretched.clip(-1.0, 1.0)
                audio_str = voc.mel_to_audio(
                    torch.from_numpy(_norm_to_logmag(stretched)).unsqueeze(0))
                str_wav = audio_dir / f'step_{step_tag}_b{i}_stretched.wav'
                sf.write(str(str_wav), audio_str, voc.h.sampling_rate)

        print(f'step {step_tag}: vocoded {mels.shape[0]} sample(s) → {audio_dir.name}/')

    # Preview latest checkpoint audio
    latest_raws = sorted(audio_dir.glob('step_*_b0_raw.wav'))
    if latest_raws:
        print(f'\nLatest raw audio ({latest_raws[-1].name}):')
        display(Audio(str(latest_raws[-1])))
    latest_strs = sorted(audio_dir.glob('step_*_b0_stretched.wav'))
    if latest_strs:
        print(f'Latest stretched audio ({latest_strs[-1].name}):')
        display(Audio(str(latest_strs[-1])))


<!-- §37-stub -->
## Cell 21 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 5.8 — DDIM denoising progression gallery
# ============================================================
# Loads best_val.pt, runs ddim_sample(return_intermediates=True),
# picks 6 evenly-spaced frames (noise → mel) and saves a 6-panel PNG.
# Saved to CKPT_DIR/denoising/{ckpt_stem}_progression.png

import pathlib, sys, csv, torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

sys.path.insert(0, '/content/MusicProject')
from omegaconf import OmegaConf
from model.unet import UNet1D
from model.diffusion import GaussianDiffusion

cfg    = OmegaConf.load('configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mc, cc, sc, dc, tc = cfg.model, cfg.conditioning, cfg.sampling, cfg.diffusion, cfg.training

# Load best checkpoint
ckpt_path = pathlib.Path(CKPT_DIR) / 'best_val.pt'
if not ckpt_path.exists():
    cks = sorted(pathlib.Path(CKPT_DIR).glob('step_*.pt'))
    assert cks, f'No checkpoints found in {CKPT_DIR}'
    ckpt_path = cks[-1]
print(f'Loading: {ckpt_path.name}')
state = torch.load(ckpt_path, map_location=device, weights_only=False)

model = UNet1D(
    mel_channels=mc.mel_channels,
    score_channels=mc.score_channels,
    base_channels=mc.base_channels,
    channel_mults=list(mc.channel_mults),
    num_res_blocks_enc=mc.num_res_blocks_enc,
    num_res_blocks_dec=mc.num_res_blocks_dec,
    attention_levels=list(mc.attention_levels),
    attn_heads=mc.attention_heads,
    n_groups=mc.n_groups,
    dropout=mc.dropout,
    n_versions=cc.n_versions,
    version_emb_dim=cc.version_emb_dim,
    time_emb_dim=cc.time_emb_dim,
).to(device).eval()

wk = 'ema' if 'ema' in state else 'model'
model.load_state_dict(state[wk], strict=True)

diffusion = GaussianDiffusion(
    model=model, T=dc.T_train, n_versions=cc.n_versions,
    cfg_score=sc.cfg_score, cfg_version=sc.cfg_version,
    cfg_drop_score=tc.cfg_drop_score, cfg_drop_version=tc.cfg_drop_version,
    cfg_drop_both=tc.cfg_drop_both,
).to(device).eval()

# Load score — prefer saved scores.pt, fall back to val CSV
sample_dir  = pathlib.Path(CKPT_DIR) / 'samples'
score_files = sorted(sample_dir.glob('step_*_scores.pt'))
if score_files:
    score = torch.load(score_files[-1], weights_only=True)[:1].to(device)  # [1, 256, 430]
    print(f'Using saved score: {score_files[-1].name}')
else:
    with open(VAL_CSV, newline='', encoding='utf-8') as f:
        row = list(csv.DictReader(f))[0]
    score_raw = torch.load(pathlib.Path(SLAKH_PROCESSED) / row['score_path'],
                           weights_only=True).unsqueeze(0).to(device)  # [1, 2, 128, 430]
    score = score_raw.view(1, -1, score_raw.shape[-1])                  # [1, 256, 430]
    print('Score loaded from val CSV (fallback — no scores.pt saved yet).')

vid = torch.zeros(1, dtype=torch.long, device=device)

print(f'Running DDIM ({sc.N_ddim} steps) with intermediates…')
with torch.no_grad():
    x0, intermediates = diffusion.ddim_sample(
        score=score, version_id=vid, N=sc.N_ddim, return_intermediates=True
    )
print(f'  x0 shape: {x0.shape}  |  {len(intermediates)} intermediate frames')

# Pick 6 evenly-spaced frames: first=noise, last=denoised
N_PANELS = 6
n_inter  = len(intermediates)
indices  = [int(round(i * (n_inter - 1) / (N_PANELS - 1))) for i in range(N_PANELS)]
frames   = [intermediates[idx] for idx in indices]
# Label with approximate diffusion timestep (decreasing)
t_labels = [f'T≈{int(round(dc.T_train * (1 - idx / (n_inter - 1))))}' for idx in indices]
t_labels[-1] = 'T=0 (output)'

denoising_dir = pathlib.Path(CKPT_DIR) / 'denoising'
denoising_dir.mkdir(exist_ok=True)
out_png = denoising_dir / f'{ckpt_path.stem}_progression.png'

fig, axes = plt.subplots(1, N_PANELS, figsize=(4 * N_PANELS, 3))
for ax, frame, label in zip(axes, frames, t_labels):
    mel_np = frame.squeeze(0).cpu().float().numpy()
    ax.imshow(mel_np, aspect='auto', origin='lower', cmap='magma', vmin=-1, vmax=1)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('time frame')
    if ax is axes[0]:
        ax.set_ylabel('mel bin')
    else:
        ax.set_yticks([])

plt.suptitle(f'DDIM Denoising Progression — {ckpt_path.stem}', fontsize=12)
plt.tight_layout()
plt.savefig(str(out_png), dpi=100)
plt.close()
print(f'Saved: {out_png}')
display(IPImage(str(out_png)))


<!-- §37-stub -->
## Cell 22 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 5.9 — MIDI piano roll visualization
# ============================================================
# Reference piano roll: always available (gate_eval/val0_reference.mid or score tensor).
# Predicted piano roll: only if *_basic_pitch.mid exists in gate_eval/.
#   → upload MIDI from local basic-pitch run (see Cell 7 instructions).
# Saved to CKPT_DIR/midi_viz/val0_piano_roll.png

import pathlib, sys, csv, torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

sys.path.insert(0, '/content/MusicProject')

midi_viz_dir = pathlib.Path(CKPT_DIR) / 'midi_viz'
midi_viz_dir.mkdir(exist_ok=True)
gate_eval = pathlib.Path(CKPT_DIR) / 'gate_eval'

# ── Reference piano roll ──────────────────────────────────────────────────
ref_midi_path = gate_eval / 'val0_reference.mid'
sr_pr = 100  # piano roll frames per second

if ref_midi_path.exists():
    import pretty_midi
    pm_ref = pretty_midi.PrettyMIDI(str(ref_midi_path))
    ref_pr = pm_ref.get_piano_roll(fs=sr_pr)   # [128, T]
    print(f'Loaded reference MIDI: {ref_midi_path.name}  ({ref_pr.shape[1]} frames)')
else:
    # Build from score tensor (sustain channel)
    print('val0_reference.mid not found — building piano roll from score tensor…')
    print('(Run Cell 7 first to generate the reference MIDI on Drive.)')
    with open(VAL_CSV, newline='', encoding='utf-8') as f:
        row = list(csv.DictReader(f))[0]
    score_np = torch.load(pathlib.Path(SLAKH_PROCESSED) / row['score_path'],
                          weights_only=True).numpy()   # [2, 128, 430]
    ref_pr = score_np[1].astype(np.float32)            # sustain channel [128, 430]
    sr_pr  = None   # frame rate unknown for score tensors

# ── Predicted piano roll (basic-pitch MIDI, if available) ─────────────────
bp_midis = sorted(gate_eval.glob('*_basic_pitch.mid')) if gate_eval.exists() else []
has_pred = bool(bp_midis)

if has_pred:
    import pretty_midi
    pm_pred = pretty_midi.PrettyMIDI(str(bp_midis[-1]))
    pred_pr = pm_pred.get_piano_roll(fs=sr_pr or 100)
    print(f'Loaded predicted MIDI: {bp_midis[-1].name}  ({pred_pr.shape[1]} frames)')
else:
    pred_pr = None
    print('No *_basic_pitch.mid found in gate_eval/ — showing reference only.')
    print(f'Upload basic-pitch MIDI to: {gate_eval}')

# ── Plot ──────────────────────────────────────────────────────────────────
def _plot_pr(ax, pr, title):
    """Plot a binary piano roll; auto-crop to active pitch range."""
    active_rows = np.where(pr.sum(axis=1) > 0)[0]
    lo = max(0,   int(active_rows.min()) - 5) if len(active_rows) else 0
    hi = min(128, int(active_rows.max()) + 6) if len(active_rows) else 128
    ax.imshow((pr[lo:hi] > 0).astype(float),
              aspect='auto', origin='lower', cmap='Blues',
              interpolation='nearest', vmin=0, vmax=1)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(f'time frame{" (~10 ms)" if sr_pr else ""}')
    ax.set_ylabel(f'MIDI pitch ({lo}–{hi})')

n_cols = 2 if has_pred else 1
fig, axes = plt.subplots(n_cols, 1, figsize=(14, 4 * n_cols))
axes = [axes] if n_cols == 1 else list(axes)

_plot_pr(axes[0], ref_pr, 'Reference piano roll')
if has_pred:
    _plot_pr(axes[1], pred_pr, f'Predicted piano roll  ({bp_midis[-1].name})')

plt.tight_layout()
out_png = midi_viz_dir / 'val0_piano_roll.png'
plt.savefig(str(out_png), dpi=100)
plt.close()
print(f'Saved: {out_png}')
display(IPImage(str(out_png)))


<!-- §37-stub -->
## Cell 23 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 5.10 — Milestone comparison (PowerPoint-ready export)
# ============================================================
# Stitches per-checkpoint mel PNGs + val-loss annotations into
# one tall figure.  Also copies individual PNGs for each milestone.
#
# Set MILESTONES to a list of step numbers, or None to use ALL
# saved checkpoints (auto-detected from mel_viz/).
#
# Output: CKPT_DIR/presentation/
#   milestone_comparison.png   ← single figure, insert into PowerPoint
#   milestone_XXXXXXX.png      ← individual slide-ready PNGs per step

import pathlib, csv, re, shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── CONFIG ────────────────────────────────────────────────────────────────
MILESTONES = None   # e.g. [30000, 60000, 90000, 100000]  or None = all
# ──────────────────────────────────────────────────────────────────────────

ppt_dir     = pathlib.Path(CKPT_DIR) / 'presentation'
mel_viz_dir = pathlib.Path(CKPT_DIR) / 'mel_viz'
loss_csv    = pathlib.Path(CKPT_DIR) / 'loss_log.csv'
ppt_dir.mkdir(exist_ok=True)

# Discover available mel PNGs
png_files = sorted(mel_viz_dir.glob('step_*.png')) if mel_viz_dir.exists() else []
available = []
for p in png_files:
    m = re.search(r'step_(\d+)', p.name)
    if m:
        available.append((int(m.group(1)), p))

if not available:
    print('No mel viz PNGs found yet.')
    print(f'They are saved during training to {mel_viz_dir}/')
    print('Train to at least one save_every checkpoint, then re-run this cell.')
else:
    # Select milestones
    if MILESTONES is not None:
        selected = []
        for ms in MILESTONES:
            closest = min(available, key=lambda x: abs(x[0] - ms))
            selected.append(closest)
            if abs(closest[0] - ms) > 1000:
                print(f'  WARN: requested {ms:,} — closest available: {closest[0]:,}')
    else:
        selected = available

    # Load val losses for annotations
    val_by_step = {}
    if loss_csv.exists():
        with open(loss_csv, newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                if row['val_loss']:
                    val_by_step[int(row['step'])] = float(row['val_loss'])

    # Build figure: one row per milestone
    n_rows = len(selected)
    fig, axes = plt.subplots(n_rows, 1, figsize=(15, 3.2 * n_rows),
                             facecolor='#0f1117')
    if n_rows == 1:
        axes = [axes]

    for ax, (step, png_path) in zip(axes, selected):
        img = mpimg.imread(str(png_path))
        ax.imshow(img)
        ax.axis('off')
        val_str = f'   val_loss = {val_by_step[step]:.4f}' if step in val_by_step else ''
        ax.set_title(f'Step {step:,}{val_str}',
                     fontsize=13, fontweight='bold', color='white', pad=5)

    plt.suptitle(f'{RUN_NAME}  —  Training Progression',
                 fontsize=16, fontweight='bold', color='white', y=1.005)
    plt.tight_layout()

    # Save combined figure (high DPI → sharp in PowerPoint)
    out_combined = ppt_dir / 'milestone_comparison.png'
    plt.savefig(str(out_combined), dpi=150, bbox_inches='tight',
                facecolor='#0f1117')
    plt.close()
    print(f'Saved: {out_combined}')
    print(f'  {n_rows} milestones: steps {[s for s, _ in selected]}')

    # Copy individual per-milestone PNGs (one per slide)
    for step, png_path in selected:
        dst = ppt_dir / f'milestone_{step:07d}.png'
        shutil.copy2(str(png_path), str(dst))
    print(f'  Individual slide PNGs → {ppt_dir}/')

    from IPython.display import Image as IPImage, display
    display(IPImage(str(out_combined), width=900))


<!-- §37-stub -->
## Cell 24 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 5.11 — Interactive HTML experiment report
# ============================================================
# Generates a single self-contained HTML file — no server needed,
# works offline, shareable by email or upload.
#
# Includes:
#   • Training loss curve
#   • Per-checkpoint mel spectrograms + audio players (raw + stretched)
#   • DDIM denoising progression gallery
#   • MIDI piano roll (reference + predicted if available)
#
# Output: CKPT_DIR/presentation/experiment_report.html
#   → Download from Drive and open in any browser.
#   → Can be embedded in a project website or shared for demo night.

import pathlib, csv, re, base64
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

ppt_dir      = pathlib.Path(CKPT_DIR) / 'presentation'
mel_viz_dir  = pathlib.Path(CKPT_DIR) / 'mel_viz'
audio_dir    = pathlib.Path(CKPT_DIR) / 'audio'
denoising_dir= pathlib.Path(CKPT_DIR) / 'denoising'
midi_viz_dir = pathlib.Path(CKPT_DIR) / 'midi_viz'
loss_csv     = pathlib.Path(CKPT_DIR) / 'loss_log.csv'
ppt_dir.mkdir(exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────
def _b64(path, mime):
    with open(path, 'rb') as f:
        return f'data:{mime};base64,{base64.b64encode(f.read()).decode()}'

def _img_tag(path, alt='', style='width:100%'):
    return f'<img src="{_b64(path, "image/png")}" alt="{alt}" style="{style}">'

def _audio_tag(path, label):
    return (f'<div class="audio-row"><span class="audio-label">{label}</span>'
            f'<audio controls src="{_b64(path, "audio/wav")}"></audio></div>')

# ── 1. Render loss curve PNG ───────────────────────────────────────────────
loss_png = ppt_dir / '_loss_curve_report.png'
if loss_csv.exists():
    train_steps, train_losses, val_steps, val_losses = [], [], [], []
    with open(loss_csv, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            s = int(row['step'])
            if row['train_loss']:
                train_steps.append(s); train_losses.append(float(row['train_loss']))
            if row['val_loss']:
                val_steps.append(s); val_losses.append(float(row['val_loss']))

    fig, ax = plt.subplots(figsize=(11, 3.5), facecolor='#1a1d27')
    ax.set_facecolor('#1a1d27')
    if train_losses:
        ax.plot(train_steps, train_losses, color='#4a9eff', lw=0.8, alpha=0.4, label='train')
        if len(train_losses) >= 20:
            k = np.ones(20) / 20
            sm = np.convolve(train_losses, k, 'valid')
            ax.plot(train_steps[19:], sm, color='#4a9eff', lw=2.0, label='train (smooth)')
    if val_losses:
        ax.plot(val_steps, val_losses, 'o-', color='#e94560', lw=1.5, ms=5, label='val loss')
        best_i = val_losses.index(min(val_losses))
        ax.axvline(val_steps[best_i], color='#e94560', ls='--', lw=0.8, alpha=0.5)
        ax.annotate(f'best {min(val_losses):.4f}',
                    xy=(val_steps[best_i], min(val_losses)),
                    xytext=(8, 8), textcoords='offset points',
                    fontsize=8, color='#e94560')
    ax.axhspan(0.15, 0.30, color='#64dfdf', alpha=0.07, label='gate zone [0.15, 0.30]')
    ax.set_xlabel('step', color='#aaa'); ax.set_ylabel('L1 loss', color='#aaa')
    ax.set_title('Training Loss', color='white', fontsize=12)
    ax.tick_params(colors='#aaa')
    for spine in ax.spines.values():
        spine.set_edgecolor('#2a2d3a')
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))
    ax.legend(fontsize=8, facecolor='#1a1d27', labelcolor='#ccc')
    ax.grid(True, lw=0.3, color='#2a2d3a')
    plt.tight_layout()
    plt.savefig(str(loss_png), dpi=120, bbox_inches='tight', facecolor='#1a1d27')
    plt.close()

# ── 2. Collect milestones ─────────────────────────────────────────────────
val_by_step = {}
if loss_csv.exists():
    with open(loss_csv, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['val_loss']:
                val_by_step[int(row['step'])] = float(row['val_loss'])

png_files = sorted(mel_viz_dir.glob('step_*.png')) if mel_viz_dir.exists() else []
milestones = []
for p in png_files:
    m = re.search(r'step_(\d+)', p.name)
    if not m:
        continue
    step = int(m.group(1))
    raw_wav  = audio_dir / f'step_{step}_b0_raw.wav'
    str_wav  = audio_dir / f'step_{step}_b0_stretched.wav'
    milestones.append({
        'step':     step,
        'mel_png':  p,
        'raw_wav':  raw_wav  if raw_wav.exists()  else None,
        'str_wav':  str_wav  if str_wav.exists()  else None,
    })

# ── 3. Build HTML sections ────────────────────────────────────────────────
sections = []

if loss_png.exists():
    sections.append(f'''
    <section class="card" id="sec-loss">
      <h2>📈 Training Loss</h2>
      {_img_tag(loss_png, "loss curve", "width:100%;max-width:860px")}
    </section>''')

if milestones:
    ms_html = []
    for ms in milestones:
        s = ms['step']
        val_tag = (f'<span class="badge">val_loss {val_by_step[s]:.4f}</span>'
                   if s in val_by_step else '')
        audio_html = ''
        if ms['raw_wav']:
            audio_html += _audio_tag(ms['raw_wav'], 'Generated (raw)')
        if ms['str_wav']:
            audio_html += _audio_tag(ms['str_wav'], 'Generated (stretched)')
        ms_html.append(f'''
        <div class="milestone">
          <h3>Step {s:,} {val_tag}</h3>
          {_img_tag(ms["mel_png"], f"mel step {s}")}
          {audio_html}
        </div>''')

    sections.append(f'''
    <section class="card" id="sec-progression">
      <h2>🎼 Training Progression — Mel Spectrograms & Audio</h2>
      {''.join(ms_html)}
    </section>''')

denoising_pngs = sorted(denoising_dir.glob('*_progression.png')) if denoising_dir.exists() else []
if denoising_pngs:
    imgs = ''.join(_img_tag(p, p.stem, 'width:100%;margin-bottom:14px')
                   for p in denoising_pngs)
    sections.append(f'''
    <section class="card" id="sec-denoising">
      <h2>🌫️ DDIM Denoising Progression (noise → mel)</h2>
      {imgs}
    </section>''')

pianoroll_png = midi_viz_dir / 'val0_piano_roll.png'
if pianoroll_png.exists():
    sections.append(f'''
    <section class="card" id="sec-midi">
      <h2>🎹 MIDI Piano Roll (reference vs. predicted)</h2>
      {_img_tag(pianoroll_png, "piano roll", "width:100%;max-width:1100px")}
    </section>''')

nav_links = ''
sec_ids   = ['sec-loss', 'sec-progression', 'sec-denoising', 'sec-midi']
sec_names = ['Loss Curve', 'Mel Progression', 'Denoising', 'Piano Roll']
for sid, sname in zip(sec_ids, sec_names):
    nav_links += f'<a href="#{sid}">{sname}</a>\n'

# ── 4. Write HTML ─────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{RUN_NAME} — Style Transfer Experiment</title>
  <style>
    *  {{ box-sizing:border-box; margin:0; padding:0; }}
    body {{ font-family:'Segoe UI',Arial,sans-serif; background:#0f1117; color:#e0e0e0; }}
    nav  {{ position:sticky; top:0; background:#16213e; padding:10px 28px;
            border-bottom:2px solid #e94560; display:flex; gap:20px; z-index:100;
            flex-wrap:wrap; align-items:center; }}
    nav span {{ font-weight:700; color:#e94560; margin-right:12px; font-size:1.05rem; }}
    nav a {{ color:#aaa; text-decoration:none; font-size:0.9rem; }}
    nav a:hover {{ color:#64dfdf; }}
    header {{ background:linear-gradient(135deg,#1a1a2e,#16213e);
              padding:44px 36px 32px; border-bottom:2px solid #e94560; }}
    header h1 {{ font-size:2.1rem; color:#fff; margin-bottom:8px; }}
    header p  {{ color:#aaa; font-size:0.95rem; line-height:1.6; }}
    main {{ max-width:1200px; margin:0 auto; padding:32px 28px; }}
    .card {{ background:#1a1d27; border-radius:12px; padding:28px;
             margin-bottom:28px; border:1px solid #2a2d3a;
             box-shadow:0 2px 16px rgba(0,0,0,0.45); }}
    .card h2 {{ font-size:1.25rem; color:#e94560; margin-bottom:20px;
               border-bottom:1px solid #2a2d3a; padding-bottom:12px; }}
    .milestone {{ margin-bottom:32px; padding-bottom:28px;
                 border-bottom:1px solid #2a2d3a; }}
    .milestone:last-child {{ border-bottom:none; margin-bottom:0; padding-bottom:0; }}
    .milestone h3 {{ font-size:1.1rem; color:#64dfdf; margin-bottom:12px; }}
    .badge {{ background:#2a2d3a; color:#aaa; font-size:0.8rem; font-weight:400;
              padding:2px 8px; border-radius:4px; margin-left:10px;
              vertical-align:middle; }}
    img  {{ border-radius:8px; display:block; }}
    .audio-row {{ display:flex; align-items:center; gap:12px;
                  margin:10px 0; flex-wrap:wrap; }}
    .audio-label {{ color:#aaa; font-size:0.85rem; min-width:180px; }}
    audio {{ accent-color:#e94560; height:32px; }}
    footer {{ text-align:center; padding:28px; color:#555; font-size:0.82rem; }}
  </style>
</head>
<body>
<nav>
  <span>🎵 {RUN_NAME}</span>
  {nav_links}
</nav>
<header>
  <h1>Style Transfer — Experiment Report</h1>
  <p>
    Run: <b>{RUN_NAME}</b> &nbsp;·&nbsp;
    {len(milestones)} checkpoints &nbsp;·&nbsp;
    Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}
  </p>
</header>
<main>
{''.join(sections)}
</main>
<footer>Generated by train_sanity.ipynb Cell 5.11 &mdash; self-contained, no server required.</footer>
</body>
</html>"""

out_html = ppt_dir / 'experiment_report.html'
out_html.write_text(html, encoding='utf-8')
size_mb = out_html.stat().st_size / 1e6
print(f'Saved: {out_html}')
print(f'  File size: {size_mb:.1f} MB  (larger = more audio embedded)')
print(f'  Sections: loss={loss_png.exists()}, '
      f'milestones={len(milestones)}, '
      f'denoising={len(denoising_pngs)}, '
      f'piano_roll={pianoroll_png.exists()}')
print()
print('To use:')
print('  1. Download from Drive → open in any browser (offline, no server needed)')
print('  2. For PowerPoint: use milestone_comparison.png or individual milestone_*.png')


<!-- §37-stub -->
## Cell 25 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
### #VSC-9f8b3270
# ============================================================
# Cell 6 — Gate check (STEP 5 part 1/2): val loss + listen
# Run after training completes
# ============================================================

import pathlib, torch
from IPython.display import Audio, display

# ── Val loss from TensorBoard logs ──────────────────────────────────────────
last_val = None
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    ea = EventAccumulator(LOG_DIR)
    ea.Reload()
    if 'val/loss' in ea.Tags()['scalars']:
        val_events = ea.Scalars('val/loss')
        last_val = val_events[-1]
        print(f'Last val/loss: {last_val.value:.4f} at step {last_val.step}')
        # Workplan gate: val_loss ∈ [0.15, 0.30]
        GATE_LO, GATE_HI = 0.15, 0.30
        if GATE_LO <= last_val.value <= GATE_HI:
            print(f'GATE PART 1 PASSED: val_loss {last_val.value:.4f} ∈ [{GATE_LO}, {GATE_HI}]')
        elif last_val.value < GATE_LO:
            print(f'WARN: val_loss {last_val.value:.4f} < {GATE_LO} (suspiciously low — check for overfitting)')
        else:
            print(f'GATE PART 1 NOT MET: val_loss {last_val.value:.4f} > {GATE_HI}  (need more training or debug)')
    else:
        print('No val/loss scalars found — training may not have reached a checkpoint yet.')
except Exception as e:
    print(f'Could not read TensorBoard logs: {e}')

# ── Listen to generated samples (sanity-only; full vocode + F1 in Cell 7) ──
sample_dir = pathlib.Path(CKPT_DIR) / 'samples'
mel_files = sorted(sample_dir.glob('step_*_mels.pt')) if sample_dir.exists() else []

if mel_files:
    latest = mel_files[-1]
    print(f'\nLatest sample mel: {latest.name}')
    mels = torch.load(latest, weights_only=True)  # [B, 80, 430]
    print(f'Generated mel shape: {mels.shape}')
    print('→ Run Cell 7 to vocode + compute F1 (full STEP 5 gate evaluation).')
else:
    print('No sample mel files found yet. Training may not have reached the first sample step.')


<!-- §37-stub -->
## Cell 26 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 5.12 — Override best_val.pt with manually chosen checkpoint
# ============================================================
# After comparing checkpoints visually (mel gallery) and by ear (audio),
# set BEST_STEP to the checkpoint that sounds/looks best.
# This overwrites best_val.pt so all downstream eval cells use it.
# Set BEST_STEP = None to keep the auto-saved best_val.pt as-is.
# ============================================================

import shutil, pathlib

BEST_STEP = 140000   # ← set to your chosen step, or None to skip

if BEST_STEP is not None:
    src = pathlib.Path(CKPT_DIR) / f'step_{BEST_STEP}.pt'
    dst = pathlib.Path(CKPT_DIR) / 'best_val.pt'
    assert src.exists(), f'Checkpoint not found: {src}'
    shutil.copy(src, dst)
    print(f'✓ best_val.pt overwritten with step_{BEST_STEP}.pt')
else:
    print('Skipped — keeping auto-saved best_val.pt')


<!-- §37-stub -->
## Cell 27 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# Cell 7 — Gate check (STEP 5 part 2/2): held-out F1 evaluation
# Vocode one val segment, transcribe with Basic-Pitch, compute F1
# Gate (REMAINING_WORKPLAN.md STEP 5):  F1 >= 0.30
# ============================================================

import csv, json, pathlib, sys
import numpy as np
import torch
import soundfile as sf
import pretty_midi
from IPython.display import Audio, display

# 1. Pick one val segment from the Slakh val split
with open(VAL_CSV, newline='', encoding='utf-8') as f:
    val_rows = list(csv.DictReader(f))
assert val_rows, 'val.csv is empty'
row = val_rows[0]

mel_pt   = pathlib.Path(SLAKH_PROCESSED) / row['segment_path']
score_pt = pathlib.Path(SLAKH_PROCESSED) / row['score_path']
print(f'Val sample : song_id={row["song_id"]}  segment={row["segment_idx"]}')
print(f'  mel path : {mel_pt}')

# 2. Load checkpoint and rebuild model + diffusion
from omegaconf import OmegaConf
sys.path.insert(0, '/content/MusicProject')
from model.unet import UNet1D
from model.diffusion import GaussianDiffusion

cfg    = OmegaConf.load('configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mc, cc, sc, dc, tc = cfg.model, cfg.conditioning, cfg.sampling, cfg.diffusion, cfg.training

ckpt_path = pathlib.Path(CKPT_DIR) / 'best_val.pt'
if not ckpt_path.exists():
    cks = sorted(pathlib.Path(CKPT_DIR).glob('step_*.pt'))
    assert cks, f'No checkpoints found in {CKPT_DIR}'
    ckpt_path = cks[-1]
print(f'Loading checkpoint: {ckpt_path.name}')
state = torch.load(ckpt_path, map_location=device, weights_only=False)

model = UNet1D(
    mel_channels=mc.mel_channels,
    score_channels=mc.score_channels,
    base_channels=mc.base_channels,
    channel_mults=list(mc.channel_mults),
    num_res_blocks_enc=mc.num_res_blocks_enc,
    num_res_blocks_dec=mc.num_res_blocks_dec,
    attention_levels=list(mc.attention_levels),
    attn_heads=mc.attention_heads,
    n_groups=mc.n_groups,
    dropout=mc.dropout,
    n_versions=cc.n_versions,
    version_emb_dim=cc.version_emb_dim,
    time_emb_dim=cc.time_emb_dim,
).to(device).eval()

weights_key = 'ema' if 'ema' in state else ('model' if 'model' in state else None)
assert weights_key, f'Unknown checkpoint format: keys={list(state.keys())}'
model.load_state_dict(state[weights_key], strict=True)
print(f'Loaded weights from key="{weights_key}".')

# Wrap in GaussianDiffusion — mirrors train.py constructor exactly
diffusion = GaussianDiffusion(
    model=model,
    T=dc.T_train,
    n_versions=cc.n_versions,
    cfg_score=sc.cfg_score,
    cfg_version=sc.cfg_version,
    cfg_drop_score=tc.cfg_drop_score,
    cfg_drop_version=tc.cfg_drop_version,
    cfg_drop_both=tc.cfg_drop_both,
).to(device).eval()

# 3. Load tensors
mel   = torch.load(mel_pt,   weights_only=True).unsqueeze(0).to(device)   # [1, 80, 430]
score = torch.load(score_pt, weights_only=True).unsqueeze(0).to(device)   # [1, 2, 128, 430]
score = score.view(1, -1, score.shape[-1])                                 # [1, 256, 430]
vid   = torch.zeros(1, dtype=torch.long, device=device)                    # version_id=0

# 4. DDIM inference  (N=100 deterministic steps)
print(f'Running DDIM ({sc.N_ddim} steps) …')
with torch.no_grad():
    gen_mel = diffusion.ddim_sample(
        score=score,
        version_id=vid,
        N=sc.N_ddim,
    )   # → [1, 80, 430] in [-1, 1]
gen_mel_cpu = gen_mel.squeeze(0).cpu()
print('Inference done.')

# Mel diagnostics: distinguish model vs vocoder corruption
_g = gen_mel_cpu.numpy()
print(f"  generated mel: min={_g.min():.3f}  max={_g.max():.3f}  mean={_g.mean():.3f}  std={_g.std():.3f}  (expected range [-1, 1])")
_t = torch.load(mel_pt, weights_only=True).numpy()
print(f"  target    mel: min={_t.min():.3f}  max={_t.max():.3f}  mean={_t.mean():.3f}  std={_t.std():.3f}")
del _t

# 5. Vocode with BigVGAN v2 22kHz
from postprocessing.bigvgan_vocoder import BigVGANVocoder
voc = BigVGANVocoder(model_name='bigvgan_22k', device=str(device))

def _norm_to_logmag(mel_norm_np):
    """Convert our training-normalized mel [-1,1] to BigVGAN log-amplitude."""
    mel_db = (mel_norm_np + 1.0) / 2.0 * 80.0 - 80.0
    return (mel_db * (np.log(10) / 20.0)).astype(np.float32)

eval_dir  = pathlib.Path(CKPT_DIR) / 'gate_eval'
eval_dir.mkdir(exist_ok=True)
_step_tag = getattr(last_val, 'step', 'final') if 'last_val' in vars() else 'final'

# ── A. Raw generated audio ───────────────────────────────────────────────
gen_np     = gen_mel_cpu.numpy()
audio_gen  = voc.mel_to_audio(torch.from_numpy(_norm_to_logmag(gen_np)).unsqueeze(0))
gen_wav    = eval_dir / f'val0_step_{_step_tag}_generated.wav'
sf.write(str(gen_wav), audio_gen, voc.h.sampling_rate)
print(f'[A] Raw generated audio: {gen_wav}')
display(Audio(str(gen_wav)))

# ── B. Contrast-stretched generated audio ────────────────────────────────
# Model at early training outputs low-variance mels (std≈0.13 vs target 0.46).
# Stretch generated to match target mel's mean/std before vocoding.
tgt_np  = torch.load(mel_pt, weights_only=True).numpy()
tgt_mean, tgt_std = float(tgt_np.mean()), float(tgt_np.std())
gen_mean, gen_std = float(gen_np.mean()), float(gen_np.std())
gen_stretched = (gen_np - gen_mean) / (gen_std + 1e-8) * tgt_std + tgt_mean
gen_stretched = gen_stretched.clip(-1.0, 1.0)
audio_stretch = voc.mel_to_audio(torch.from_numpy(_norm_to_logmag(gen_stretched)).unsqueeze(0))
stretch_wav   = eval_dir / f'val0_step_{_step_tag}_generated_stretched.wav'
sf.write(str(stretch_wav), audio_stretch, voc.h.sampling_rate)
print(f'[B] Contrast-stretched generated: {stretch_wav}')
print(f'    (stretched std {gen_std:.3f}→{tgt_std:.3f}, mean {gen_mean:.3f}→{tgt_mean:.3f})')
display(Audio(str(stretch_wav)))

# ── C. Target mel re-vocoded (ground truth quality check) ────────────────
audio_tgt  = voc.mel_to_audio(torch.from_numpy(_norm_to_logmag(tgt_np)).unsqueeze(0))
tgt_wav    = eval_dir / 'val0_target_vocoded.wav'
sf.write(str(tgt_wav), audio_tgt, voc.h.sampling_rate)
print(f'[C] Target mel re-vocoded (reference): {tgt_wav}')
display(Audio(str(tgt_wav)))

# Keep gen_wav pointing to the raw file (used by Cell 7 instructions below)

# 6. Build reference MIDI from piano roll
ref_midi = eval_dir / 'val0_reference.mid'
pr_seg   = torch.load(score_pt, weights_only=True).numpy()   # [2, 128, 430]
onset, sustain = pr_seg[0], pr_seg[1]
pm   = pretty_midi.PrettyMIDI()
inst = pretty_midi.Instrument(program=0)
fps  = 22050.0 / 256.0
for pitch in range(128):
    active = sustain[pitch] > 0
    if not active.any():
        continue
    in_note = False
    start_t = 0.0
    for f, on in enumerate(active):
        if on and not in_note:
            in_note = True
            start_t = f / fps
        elif (not on) and in_note:
            in_note = False
            inst.notes.append(pretty_midi.Note(velocity=100, pitch=pitch,
                                                start=start_t, end=f / fps))
    if in_note:
        inst.notes.append(pretty_midi.Note(velocity=100, pitch=pitch,
                                            start=start_t, end=len(active) / fps))
pm.instruments.append(inst)
pm.write(str(ref_midi))
print(f'Reference MIDI: {ref_midi}  ({sum(len(i.notes) for i in pm.instruments)} notes)')


# 7. Local basic-pitch transcription required
# ─────────────────────────────────────────────────────────────────────
# Colab runs Python 3.12; basic-pitch requires Python ≤3.10.
# Run basic-pitch locally on the generated WAV, upload the MIDI to Drive,
# then run Cell 7b below to compute F1 and check the gate.
#
# ── LOCAL COMMAND (Windows, in MusicProject folder) ──────────────────
_wav_local = str(gen_wav).replace('/content/drive/MyDrive/', 'G:\\My Drive\\').replace('/', '\\')
_bp_midi_drive = gen_wav.parent / f'{gen_wav.stem}_basic_pitch.mid'
print()
print('=' * 65)
print('ACTION REQUIRED: run basic-pitch locally then run Cell 7b')
print('=' * 65)
print()
print('1. On your local Windows machine (MusicProject folder):')
print(f'   basic_pitch_env\\Scripts\\basic-pitch midi_output "{_wav_local}"')
print()
print('2. Copy the output MIDI to Drive at:')
print(f'   {_bp_midi_drive}')
print()
print('3. Run Cell 7b.')


<!-- §37-stub -->
## Cell 28 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# Cell 7b: F1 gate check
# Run this after uploading the basic-pitch MIDI to Drive (see Cell 7 instructions).
import pathlib, json
from postprocessing.f1_eval import compute_f1_from_midi

eval_dir = pathlib.Path(CKPT_DIR) / 'gate_eval'
ref_midi = eval_dir / 'val0_reference.mid'

bp_midi_candidates = sorted(eval_dir.glob('*_basic_pitch.mid'))
assert bp_midi_candidates, (
    f'No *_basic_pitch.mid found in {eval_dir}.\n'
    'Run Cell 7 first, then basic-pitch locally, then upload the MIDI to Drive.'
)
bp_midi = bp_midi_candidates[-1]
print(f'Using predicted MIDI: {bp_midi.name}')

result = compute_f1_from_midi(
    predicted_midi=str(bp_midi),
    reference_midi=str(ref_midi),
    onset_tolerance_s=0.05,
)
print(json.dumps(result, indent=2))

with open(eval_dir / 'gate_result.json', 'w') as f:
    json.dump(result, f, indent=2)

GATE_F1 = 0.30
if result['f1'] >= GATE_F1:
    print(f'GATE PART 2 PASSED: F1 {result["f1"]:.3f} >= {GATE_F1}')
else:
    print(f'GATE PART 2 NOT MET: F1 {result["f1"]:.3f} < {GATE_F1}')

# ── STEP 5 GATE SUMMARY ──────────────────────────────────────────────────────
print()
print('=' * 60)
print('STEP 5 GATE SUMMARY  (sanity training — single style)')
print('=' * 60)
_lv = getattr(last_val, 'value', None) if 'last_val' in vars() else None
print(f'  val_loss = {_lv:.4f}   (gate: ∈ [0.15, 0.30])' if _lv is not None else '  val_loss = (run Cell 6 first)')
print(f'  F1       = {result["f1"]:.3f}    (gate: >= {GATE_F1})')
print()
print('LISTEN CHECKLIST (subjective — all 3 must pass):')
print('  [ ] Output is musical (not noise/silence/static)')
print('  [ ] Pitched notes are audible and roughly match the piano roll')
print('  [ ] Output sounds like rock instrumentation (not random timbre)')
print()
print('NOTE — what this sanity run can and cannot prove:')
print('  ✓  Model can reconstruct rock audio from a piano roll (version_id=0)')
print('  ✓  DDIM sampling produces coherent mel spectrograms')
print('  ✗  Style TRANSFER not testable yet — only one style (rock) was trained')
print('     Style transfer requires Phase 2: add Israeli music as version_id=1,')
print('     then test same piano roll → version_id=0 (rock) vs version_id=1 (Israeli).')
print()
print('If ALL checks pass → proceed to STEP 6 / STEP 7 (Israeli music training).')
